# pyDBSI — La matematica del toolbox in figure

Compagno visivo del report *Il viaggio del voxel*: per ogni passaggio che il report scrive in formule, qui
c'è una figura. I numeri delle sezioni (§) sono quelli del report.

**Da dove vengono le figure.** Il più possibile dal toolbox stesso: il notebook lancia il fit vero e ne
**registra** gli ingressi (la matrice A, i λ, il dizionario, il segnale corretto) e ogni passo della
calibrazione. Dove una figura mostra un passaggio interno che il toolbox non restituisce (i bacini, i
candidati del cono, la traiettoria del Levenberg–Marquardt) lo ricalcola con le stesse funzioni e **controlla**
che il risultato coincida con quello del toolbox: se non coincide, la cella si ferma.

**Dati.** Le sezioni sulla calibrazione (§7–9) girano su un'acquisizione vera del P3 se le trovi i file
(configurazione qui sotto: la sessione P3 dei 5P sul NAS), altrimenti sul fantoccio. Le sezioni che hanno
bisogno della verità (§14, §17, §18, §23) girano sul fantoccio della batteria.

**Quanto dura.** ~1–2 ore, quasi tutte nella §7 (stabilità di λ al variare del campione: 4 dimensioni × 5
semi = 20 calibrazioni). `STABILITA_LAMBDA = False` la salta. Le figure finiscono in `risultati_matematica/`.

## 0 — Configurazione

In [ ]:
STABILITA_LAMBDA = True          # §7: 20 calibrazioni (la parte lunga)
N_LIST_LAMBDA    = [250, 500, 1000, 2000]
SEMI_LAMBDA      = [0, 1, 2, 3, 4]
SNR_FANTOCCIO    = 26

# Un'acquisizione vera del P3 per la calibrazione (§7-9). Se un file manca si usa il fantoccio.
REALE = dict(
    prep = "/mnt/msbiobnas/Users/Francesco/DBSI_FG/DATA_5P/Processed/000003/20251210/DWI_4s/prep",
    stem = "DWITE106P2sense19nsoppyAPP",
)

## 1 — Il codice (fantocci e figure)

Sorgenti in `notebooks/sorgenti/` (`sim_accuratezza.py`, `mat_figure.py`); si rigenera con `python notebooks/sorgenti/build_matematica.py`.

In [ ]:
"""
Simulazioni di accuratezza per pyDBSI (>= 1.9.0): quanto le mappe del toolbox si avvicinano alla verita'.

Tutto passa per la CATENA DI PRODUZIONE, non per una scorciatoia:

    segnale vero (S0 = 1000)  ->  rumore Rician (sigma = S0 / SNR)  ->  dwidenoise (MRtrix, MP-PCA)
        ->  mappa -noise di dwidenoise come noise_map  ->  DBSI_Adaptive().fit(run_calibration=True)

cioe' cio' che succede ai 5P (calibrazione libera, mappa di rumore di dwidenoise). Il braccio "grezzo"
salta il denoising e passa la sigma vera: misura lo stimatore da solo, e la differenza fra i due bracci
e' l'effetto del denoising. Non c'e' N4: il fantoccio non ha campo di bias.

Tre fantocci, tre domande:
  - BATTERIA     13 scenari omogenei (quelli della validazione di luglio), blocchi 3D con eterogeneita'
                 lieve: per ogni scenario, bias e dispersione di ogni mappa.
  - ANATOMICO    una fetta (48x48x6) "a forma di cervello" (GM, due fasci che si incrociano a 90 gradi, un terzo a
                 60, ventricolo, lesioni demielinizzante / edematosa / infiammatoria, tumore) con volume
                 parziale ai bordi: le immagini vero / stimato / errore.
  - CAMPI        parametri che variano con continuita' nello spazio (campi gaussiani lisci): stima contro
                 verita' voxel per voxel, su tutto l'intervallo dei valori.

La verita' delle diffusivita' isotrope NON coincide coi centroidi fissi di Stage D (0.15 / 1.0 / 3.0e-3)
se D_VARIABILE: R in [0.10, 0.25], H in [0.7, 1.3], W in [2.8, 3.2] e-3 mm^2/s, estratte per voxel. E' il
caso reale (il tessuto non ha diffusivita' esattamente uguali ai centroidi); con D_VARIABILE=False si
misura lo stimatore nel caso in cui il modello e' esatto.
"""

import contextlib
import io
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import dbsi_toolbox
from dbsi_toolbox import DBSI_Adaptive

S0 = 1000.0
D_NOM = dict(R=0.15e-3, H=1.0e-3, W=3.0e-3)
D_RANGE = dict(R=(0.10e-3, 0.25e-3), H=(0.7e-3, 1.3e-3), W=(2.8e-3, 3.2e-3))

# metriche riportate: nome breve -> canale del toolbox
METRICHE = dict(FF='fiber_fraction', RF='restricted_fraction', HF='hindered_fraction',
                WF='water_fraction', ADw='axial_diffusivity_weighted',
                RDw='radial_diffusivity_weighted', FAw='fiber_fa_weighted')
FRAZIONI = ('FF', 'RF', 'HF', 'WF')
TENSORE = ('ADw', 'RDw', 'FAw')
UNITA = dict(FF='', RF='', HF='', WF='', ADw='e-3 mm²/s', RDw='e-3 mm²/s', FAw='')
SCALA = dict(FF=1, RF=1, HF=1, WF=1, ADw=1e3, RDw=1e3, FAw=1)    # diffusivita' mostrate in e-3

NOMI = DBSI_Adaptive.output_map_names(3)
IX = {n: i for i, n in enumerate(NOMI)}


# ─────────────────────────────────────────────────────────────────────────────
# protocollo
# ─────────────────────────────────────────────────────────────────────────────
def carica_protocollo(bval, bvec):
    """bval/bvec in formato FSL; bvec normalizzati, righe (N, 3)."""
    bv = np.loadtxt(bval).ravel().astype(float)
    bc = np.loadtxt(bvec)
    bc = bc.T if bc.shape[0] == 3 and bc.shape[1] != 3 else bc
    n = np.linalg.norm(bc, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return bv, bc / n


def descrivi_protocollo(prot):
    bv = prot[0]
    gusci = pd.Series(np.round(bv, -2).astype(int)).value_counts().sort_index()
    return ', '.join(f"b={b}: {n}" for b, n in gusci.items())


# ─────────────────────────────────────────────────────────────────────────────
# tessuto -> segnale e verita'
# ─────────────────────────────────────────────────────────────────────────────
def direzione(theta_deg, phi_deg):
    t, p = np.radians(theta_deg), np.radians(phi_deg)
    v = np.array([np.sin(t) * np.cos(p), np.sin(t) * np.sin(p), np.cos(t)])
    return v / np.linalg.norm(v)


def ruota_verso(u, gradi, asse_aux=(0.0, 0.0, 1.0)):
    """Direzione a `gradi` gradi da u, nel piano di u e di asse_aux."""
    a = np.asarray(asse_aux, float)
    t = a - (a @ u) * u
    if np.linalg.norm(t) < 1e-6:
        t = np.cross(u, [1.0, 0, 0])
    t /= np.linalg.norm(t)
    g = np.radians(gradi)
    return np.cos(g) * u + np.sin(g) * t


def segnale(prot, fibre, iso, D):
    """fibre: [(f, dir, AD, RD)], iso: {'R': f, 'H': f, 'W': f}, D: {'R': D, ...}. S(b=0) = somma f."""
    bv, bc = prot
    S = np.zeros(len(bv))
    for k, f in iso.items():
        if f > 0:
            S += f * np.exp(-bv * D[k])
    for f, d, ad, rd in fibre:
        if f > 0:
            S += f * np.exp(-bv * (rd + (ad - rd) * (bc @ d) ** 2))
    return S


def fa(ad, rd):
    return np.abs(ad - rd) / np.sqrt(ad ** 2 + 2 * rd ** 2)


SOGLIA_POP = 0.05     # una popolazione esiste, per la verita', se pesa almeno il 5% del voxel


def verita(fibre, iso, soglia_pop=SOGLIA_POP):
    """Valori veri delle metriche riportate (frazioni normalizzate al totale). Il tensore vero e' definito
    solo dalle popolazioni sopra soglia: una coda di volume parziale da 1% non ha un tensore misurabile."""
    tot = sum(f for f, *_ in fibre) + sum(iso.values())
    ff = sum(f for f, *_ in fibre)
    vive = [x for x in fibre if x[0] / tot >= soglia_pop]
    t = dict(FF=ff / tot, RF=iso.get('R', 0) / tot, HF=iso.get('H', 0) / tot, WF=iso.get('W', 0) / tot,
             npop=len(vive))
    fv = sum(f for f, *_ in vive)
    if fv > 0:
        adw = sum(f * ad for f, _, ad, _ in vive) / fv
        rdw = sum(f * rd for f, _, _, rd in vive) / fv
        t.update(ADw=adw, RDw=rdw, FAw=fa(adw, rdw))
    else:
        t.update(ADw=np.nan, RDw=np.nan, FAw=np.nan)
    return t


def estrai_D(rng, variabile):
    if not variabile:
        return dict(D_NOM)
    return {k: rng.uniform(*D_RANGE[k]) for k in D_NOM}


# ─────────────────────────────────────────────────────────────────────────────
# BATTERIA: 13 scenari (validazione di luglio), blocchi 3D omogenei
# ─────────────────────────────────────────────────────────────────────────────
_d0, _d1, _d2 = direzione(90, 0), direzione(65, 0), direzione(65, 90)


def _I(r, h, w):
    return dict(R=r, H=h, W=w)


SCENARI = {
    'WM sana':          ([(0.55, _d0, 1.7e-3, 0.30e-3)], _I(0.10, 0.25, 0.10)),
    'WM demielinizzata': ([(0.45, _d0, 1.5e-3, 0.70e-3)], _I(0.15, 0.25, 0.15)),
    'WM densa':         ([(0.70, _d0, 1.8e-3, 0.25e-3)], _I(0.10, 0.15, 0.05)),
    'WM debole':        ([(0.20, _d0, 1.7e-3, 0.30e-3)], _I(0.10, 0.50, 0.20)),
    'Crossing bil.':    ([(0.35, _d1, 1.6e-3, 0.40e-3), (0.35, _d2, 1.6e-3, 0.40e-3)], _I(0.10, 0.15, 0.05)),
    'Crossing sbil.':   ([(0.45, _d1, 1.7e-3, 0.30e-3), (0.20, _d2, 1.6e-3, 0.40e-3)], _I(0.10, 0.15, 0.10)),
    'Crossing eterog.': ([(0.30, _d1, 1.7e-3, 0.30e-3), (0.20, _d2, 1.4e-3, 0.65e-3)], _I(0.18, 0.14, 0.18)),
    'GM':               ([(0.10, _d0, 1.5e-3, 0.50e-3)], _I(0.10, 0.65, 0.15)),
    'Liquor':           ([], _I(0.0, 0.05, 0.95)),
    'Solo restricted':  ([], _I(0.30, 0.50, 0.20)),
    'Tumore':           ([], _I(0.55, 0.30, 0.15)),
    'Edema':            ([(0.30, _d0, 1.7e-3, 0.35e-3)], _I(0.0, 0.15, 0.55)),
    'Lesione infiamm.': ([(0.25, _d0, 1.6e-3, 0.45e-3)], _I(0.35, 0.30, 0.10)),
}


def fantoccio_batteria(prot, blocco=(8, 8, 6), colonne=7, eterog=0.05, d_variabile=True, seed=0, scenari=None):
    """Un blocco 3D per scenario, su una griglia di `colonne` blocchi. Eterogeneita' lieve nel blocco:
    frazioni x(1 + N(0, eterog)), direzioni che ruotano di qualche grado con la posizione, D isotrope per
    voxel. Ritorna (A pulito in unita' di segnale, verita' {metrica: (X,Y,Z)}, etichette (X,Y,Z) int con -1
    fuori dai blocchi, nomi)."""
    scenari = scenari or SCENARI
    rng = np.random.default_rng(seed)
    bx, by, bz = blocco
    nomi = list(scenari)
    righe = int(np.ceil(len(nomi) / colonne))
    X, Y, N = bx * colonne, by * righe, len(prot[0])
    A = np.zeros((X, Y, bz, N), np.float32)
    lab = np.full((X, Y, bz), -1, np.int16)
    tru = {k: np.full((X, Y, bz), np.nan) for k in (*METRICHE, 'npop')}
    tru['dirs'] = np.full((X, Y, bz, 2, 3), np.nan)
    for s, nome in enumerate(nomi):
        x0, y0 = (s % colonne) * bx, (s // colonne) * by
        fibre0, iso0 = scenari[nome]
        for i in range(bx):
            for j in range(by):
                for k in range(bz):
                    jit = 1 + rng.normal(0, eterog, 3 + len(fibre0))
                    iso = {c: iso0[c] * jit[q] for q, c in enumerate('RHW')}
                    fibre = []
                    for q, (f, d, ad, rd) in enumerate(fibre0):
                        dd = ruota_verso(d, 4.0 * (i - bx / 2) / bx + 3.0 * (j - by / 2) / by)
                        fibre.append((f * jit[3 + q], dd, ad, rd))
                    D = estrai_D(rng, d_variabile)
                    S = segnale(prot, fibre, iso, D)
                    tot = sum(f for f, *_ in fibre) + sum(iso.values())
                    x, y = x0 + i, y0 + j
                    A[x, y, k] = S / tot
                    lab[x, y, k] = s
                    for m, v in verita(fibre, iso).items():
                        tru[m][x, y, k] = v
                    for q, (_, dd, _, _) in enumerate(fibre[:2]):
                        tru['dirs'][x, y, k, q] = dd
    return A, tru, lab, nomi


# ─────────────────────────────────────────────────────────────────────────────
# ANATOMICO: una fetta con fasci, incroci, ventricolo e lesioni, volume parziale ai bordi
# ─────────────────────────────────────────────────────────────────────────────
def _morbida(dist, larghezza=0.8):
    """Appartenenza in [0, 1] da una distanza con segno (negativa dentro)."""
    return 1.0 / (1.0 + np.exp(dist / larghezza))


TESSUTI_ANAT = {
    # nome: (fibre come [(f, chiave_direzione, AD, RD)], iso)
    'GM':              ([], _I(0.10, 0.75, 0.15)),
    'fascio A':        ([(0.60, 'A', 1.7e-3, 0.30e-3)], _I(0.10, 0.22, 0.08)),
    'fascio B':        ([(0.60, 'B', 1.7e-3, 0.30e-3)], _I(0.10, 0.22, 0.08)),
    'fascio C':        ([(0.60, 'C', 1.7e-3, 0.35e-3)], _I(0.10, 0.22, 0.08)),
    'liquor':          ([], _I(0.0, 0.05, 0.95)),
    'demielinizzata':  ([(0.45, 'A', 1.5e-3, 0.75e-3)], _I(0.10, 0.35, 0.10)),
    'edema':           ([(0.30, 'B', 1.7e-3, 0.35e-3)], _I(0.05, 0.20, 0.45)),
    'infiammazione':   ([(0.40, 'A', 1.65e-3, 0.40e-3)], _I(0.30, 0.22, 0.08)),
    'tumore':          ([], _I(0.50, 0.35, 0.15)),
}


def fantoccio_anatomico(prot, nx=64, ny=64, nz=6, d_variabile=True, seed=1):
    """Ritorna (A, verita', pesi {tessuto: (X,Y,Z)}, direzioni vere per voxel {chiave: (X,Y,Z,3)})."""
    rng = np.random.default_rng(seed)
    x, y = np.meshgrid(np.arange(nx) + 0.5, np.arange(ny) + 0.5, indexing='ij')
    c = np.array([nx / 2, ny / 2])
    # fascio A: banda orizzontale leggermente curva; B: banda verticale (incrocio a 90 con A);
    # C: banda obliqua a 60 gradi rispetto ad A, nella meta' bassa
    yA = ny * 0.62 + 4 * np.sin(2 * np.pi * x / nx)
    m = {}
    m['fascio A'] = _morbida(np.abs(y - yA) - 6)
    m['fascio B'] = _morbida(np.abs(x - nx * 0.30) - 5)
    ang = np.radians(60)
    dC = np.abs((x - nx * 0.70) * np.sin(ang) - (y - ny * 0.25) * np.cos(ang))
    m['fascio C'] = _morbida(dC - 4.5) * _morbida(y - ny * 0.50, 1.5)
    m['liquor'] = _morbida(np.hypot((x - nx * 0.55) / 7, (y - ny * 0.40) / 4) * 4 - 4)
    def disco(cx, cy, r):
        return _morbida(np.hypot(x - cx, y - cy) - r)
    m['demielinizzata'] = disco(nx * 0.80, ny * 0.62 + 4 * np.sin(2 * np.pi * 0.80), 4.0)    # sul fascio A
    m['infiammazione'] = disco(nx * 0.55, ny * 0.62 + 4 * np.sin(2 * np.pi * 0.55), 3.5)     # sul fascio A
    m['edema'] = disco(nx * 0.30, ny * 0.15, 4.0)                                          # sul fascio B
    m['tumore'] = disco(nx * 0.15, ny * 0.40, 4.5)
    # composizione dall'alto (alpha compositing): liquor > lesioni > fasci > GM. I fasci si dividono
    # la copertura dove si sovrappongono: e' li' che nasce l'incrocio.
    w = {k: np.zeros((nx, ny)) for k in TESSUTI_ANAT}
    resto = np.ones((nx, ny))
    for k in ('liquor', 'tumore', 'demielinizzata', 'infiammazione', 'edema'):
        w[k] = resto * m[k]
        resto = resto - w[k]
    fasci = ('fascio A', 'fascio B', 'fascio C')
    somma = sum(m[k] for k in fasci)
    cop = np.minimum(somma, 1.0)
    for k in fasci:
        w[k] = resto * cop * np.where(somma > 0, m[k] / np.maximum(somma, 1e-12), 0)
    w['GM'] = resto * (1 - cop)
    # direzioni: A segue la curva, B verticale, C a 60 gradi da x; inclinazione fuori piano 15 gradi
    dirs = {}
    slopeA = 4 * 2 * np.pi / nx * np.cos(2 * np.pi * x / nx)
    a = np.arctan(slopeA)
    el = np.radians(15)
    dirs['A'] = np.stack([np.cos(a) * np.cos(el), np.sin(a) * np.cos(el), np.full_like(a, np.sin(el))], -1)
    dirs['B'] = np.broadcast_to(direzione(90 - 15, 90), (nx, ny, 3)).copy()
    dirs['C'] = np.broadcast_to(direzione(90 - 15, 60), (nx, ny, 3)).copy()
    N = len(prot[0])
    A = np.zeros((nx, ny, nz, N), np.float32)
    tru = {k: np.full((nx, ny, nz), np.nan) for k in (*METRICHE, 'npop')}
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                D = estrai_D(rng, d_variabile)
                fib_tot, iso_tot = {}, dict(R=0.0, H=0.0, W=0.0)
                for t, wt in w.items():
                    ww = wt[i, j]
                    if ww < 1e-4:
                        continue
                    fl, iso = TESSUTI_ANAT[t]
                    jit = 1 + rng.normal(0, 0.04, 3)
                    for q, cc in enumerate('RHW'):
                        iso_tot[cc] += ww * iso[cc] * jit[q]
                    for f, key, ad, rd in fl:
                        a0 = fib_tot.get(key, [0.0, 0.0, 0.0])
                        fib_tot[key] = [a0[0] + ww * f, a0[1] + ww * f * ad, a0[2] + ww * f * rd]
                fibre = [(v[0], dirs[key][i, j], v[1] / v[0], v[2] / v[0]) for key, v in fib_tot.items()
                         if v[0] > 1e-4]
                S = segnale(prot, fibre, iso_tot, D)
                tot = sum(f for f, *_ in fibre) + sum(iso_tot.values())
                A[i, j, k] = S / tot
                for mm, v in verita(fibre, iso_tot).items():
                    tru[mm][i, j, k] = v
    pesi = {t: np.repeat(wt[:, :, None], nz, 2) for t, wt in w.items()}
    return A, tru, pesi


# ─────────────────────────────────────────────────────────────────────────────
# CAMPI: parametri lisci e continui
# ─────────────────────────────────────────────────────────────────────────────
def _campo(rng, shape, scala):
    from scipy.ndimage import gaussian_filter
    g = gaussian_filter(rng.normal(size=shape), scala, mode='wrap')
    g = (g - g.mean()) / (g.std() + 1e-12)
    from scipy.stats import norm
    return norm.cdf(g)              # uniforme in (0, 1), liscio


def fantoccio_campi(prot, shape=(48, 48, 8), scala=3.0, d_variabile=True, seed=2):
    """FF, frazioni iso, AD, RD, angolo di incrocio e quota della seconda fibra variano con continuita'.
    Meta' del volume ha due fibre (incrocio con angolo fra 45 e 90 gradi), meta' una."""
    rng = np.random.default_rng(seed)
    u = {k: _campo(rng, shape, scala) for k in ('ff', 'r', 'h', 'w', 'ad', 'rd', 'th', 'ph', 'ang', 'q2', 'cross')}
    N = len(prot[0])
    A = np.zeros(shape + (N,), np.float32)
    tru = {k: np.full(shape, np.nan) for k in (*METRICHE, 'npop')}
    for idx in np.ndindex(shape):
        g = {k: v[idx] for k, v in u.items()}
        ff = 0.75 * g['ff']                                   # 0 - 0.75
        wiso = np.array([0.05 + g['r'], 0.05 + 1.5 * g['h'], 0.02 + 0.8 * g['w']])
        wiso = wiso / wiso.sum() * (1 - ff)
        ad = 1.3e-3 + 0.7e-3 * g['ad']
        rd = min(0.15e-3 + 0.65e-3 * g['rd'], ad / 1.6)
        d1 = direzione(30 + 120 * g['th'], 360 * g['ph'])
        fibre = []
        if ff > 0.03:
            if g['cross'] > 0.5:
                d2 = ruota_verso(d1, 45 + 45 * g['ang'], asse_aux=direzione(90, 360 * g['ph'] + 90))
                q2 = 0.3 + 0.4 * g['q2']
                fibre = [(ff * (1 - q2), d1, ad, rd), (ff * q2, d2, ad, rd)]
            else:
                fibre = [(ff, d1, ad, rd)]
        iso = dict(R=wiso[0], H=wiso[1], W=wiso[2])
        D = estrai_D(rng, d_variabile)
        S = segnale(prot, fibre, iso, D)
        A[idx] = S / (sum(f for f, *_ in fibre) + wiso.sum())
        for m, v in verita(fibre, iso).items():
            tru[m][idx] = v
    return A, tru


# ─────────────────────────────────────────────────────────────────────────────
# VOLUME UNICO: i tre fantocci affiancati, una calibrazione e un dwidenoise per condizione
# ─────────────────────────────────────────────────────────────────────────────
def volume_composito(prot, d_variabile=True, seed=0, piccolo=False, spazio=4):
    """Anatomico | campi | batteria affiancati lungo x, separati da `spazio` voxel di fondo (segnale
    nullo, fuori maschera). La calibrazione campiona i voxel di tutti e tre, come in un cervello con molti
    tessuti; dwidenoise vede un volume unico. piccolo=True dimezza i fantocci (prova veloce).
    Ritorna (A, maschera, parti) con parti[nome] = dict(sl=slice lungo x, verita'=..., ...)."""
    if piccolo:
        an = fantoccio_anatomico(prot, nx=40, ny=40, nz=6, d_variabile=d_variabile, seed=seed + 1)
        ca = fantoccio_campi(prot, shape=(28, 28, 6), d_variabile=d_variabile, seed=seed + 2)
        ba = fantoccio_batteria(prot, blocco=(6, 6, 6), colonne=7, d_variabile=d_variabile, seed=seed)
    else:
        an = fantoccio_anatomico(prot, nx=48, ny=48, nz=6, d_variabile=d_variabile, seed=seed + 1)
        ca = fantoccio_campi(prot, shape=(32, 32, 6), d_variabile=d_variabile, seed=seed + 2)
        ba = fantoccio_batteria(prot, blocco=(8, 8, 6), colonne=7, d_variabile=d_variabile, seed=seed)
    pezzi = [('anatomico', an[0]), ('campi', ca[0]), ('batteria', ba[0])]
    Y = max(p.shape[1] for _, p in pezzi)
    Z = pezzi[0][1].shape[2]
    X = sum(p.shape[0] for _, p in pezzi) + spazio * (len(pezzi) + 1)
    N = len(prot[0])
    A = np.zeros((X, Y, Z, N), np.float32)
    mask = np.zeros((X, Y, Z), bool)
    parti, x0 = {}, spazio
    for nome, P in pezzi:
        nx, ny = P.shape[:2]
        sl = (slice(x0, x0 + nx), slice(0, ny))
        A[sl] = P
        mask[sl] = P.max(-1) > 0          # il posto vuoto della griglia della batteria resta fuori
        parti[nome] = dict(sl=sl)
        x0 += nx + spazio
    parti['anatomico'].update(verita=an[1], pesi=an[2])
    parti['campi'].update(verita=ca[1])
    parti['batteria'].update(verita=ba[1], etichette=ba[2], nomi=ba[3])
    return A, mask, parti


def ritaglia(mappe, sl):
    """Le mappe stimate ({nome: (X,Y,Z)}) ritagliate su una parte del volume."""
    return {k: v[sl] for k, v in mappe.items()}


# ─────────────────────────────────────────────────────────────────────────────
# catena: rumore -> dwidenoise -> fit
# ─────────────────────────────────────────────────────────────────────────────
def rumore_rician(A, snr, seed=0):
    rng = np.random.default_rng(seed)
    sig = S0 / snr
    X = S0 * A.astype(np.float64)
    M = np.sqrt((X + rng.normal(0, sig, X.shape)) ** 2 + rng.normal(0, sig, X.shape) ** 2)
    return M.astype(np.float32), sig


def trova_dwidenoise(percorso=None):
    for p in (percorso, shutil.which('dwidenoise'), '/usr/local/mrtrix3/dwidenoise',
              '/usr/local/mrtrix3/bin/dwidenoise'):
        if p and os.path.isfile(p) and os.access(p, os.X_OK):
            return p
    return None


def denoise(M, eseguibile, voxel_mm=2.0):
    """dwidenoise di MRtrix sui dati (X,Y,Z,N): ritorna (denoisati, mappa -noise)."""
    import nibabel as nib
    aff = np.diag([voxel_mm, voxel_mm, voxel_mm, 1.0])
    with tempfile.TemporaryDirectory() as td:
        fi, fo, fn = (os.path.join(td, n) for n in ('in.nii', 'den.nii', 'noise.nii'))
        nib.save(nib.Nifti1Image(M.astype(np.float32), aff), fi)
        subprocess.run([eseguibile, fi, fo, '-noise', fn, '-force', '-quiet'], check=True)
        D = np.asarray(nib.load(fo).dataobj, dtype=np.float32)
        nm = np.asarray(nib.load(fn).dataobj, dtype=np.float64)
    return D, nm


# Un braccio = la catena del rumore ('grezzo' o 'produzione') + le opzioni del modello ({} = i default del
# toolbox installato). 'grezzo_18' = la 1.9.0 con le due opzioni dei crossing spente, cioe' il fit della 1.8.
# braccio_supportato() dice prima di fittare se il toolbox installato conosce le opzioni di un braccio.
BRACCI_DEF = {
    'grezzo':     ('grezzo', {}),
    'produzione': ('produzione', {}),
    'grezzo_18':  ('grezzo', dict(crossing_dir_refine=False, two_vs_one_threshold=None)),
}


def braccio_supportato(braccio):
    """Il toolbox installato accetta le opzioni del braccio? (False = opzioni che non conosce)."""
    try:
        DBSI_Adaptive(**BRACCI_DEF[braccio][1])
    except TypeError:
        return False
    return True


def fit(data, prot, noise_map, mask=None, opzioni=None, **kw_fit):
    """Fit di produzione (calibrazione libera salvo kw_fit; opzioni = argomenti di DBSI_Adaptive, vuoto =
    default): ritorna (risultati, modello)."""
    mask = np.ones(data.shape[:3], bool) if mask is None else mask
    m = DBSI_Adaptive(**(opzioni or {}))
    kw = dict(run_calibration=True)
    kw.update(kw_fit)
    with contextlib.redirect_stdout(io.StringIO()):
        res, mode = m.fit(data, prot[0], prot[1], mask, noise_map=noise_map, **kw)
    assert mode == 3, f'modo {mode}-ISO: il protocollo non regge il 3-ISO'
    return res, m


def esegui(A, prot, snr, braccio, dwidenoise=None, seed=0, mask=None, **kw_fit):
    """Un braccio della catena (chiave di BRACCI_DEF). Catena 'produzione' = dwidenoise + sua mappa, 'grezzo'
    = sigma vera; le opzioni del braccio vanno al modello. Il rumore e' aggiunto a TUTTO il volume, fondo
    compreso (come in uno scanner); il fit solo nella maschera. Stesso seme = stesso rumore in ogni braccio."""
    catena, opzioni = BRACCI_DEF[braccio]
    M, sig = rumore_rician(A, snr, seed)
    if catena == 'produzione':
        if dwidenoise is None:
            raise RuntimeError('dwidenoise non trovato: braccio di produzione impossibile')
        D, nm = denoise(M, dwidenoise)
        data, noise = D, nm
    elif catena == 'grezzo':
        data, noise = M, np.full(A.shape[:3], sig)
    else:
        raise ValueError(catena)
    res, m = fit(data, prot, noise, mask=mask, opzioni=opzioni, **kw_fit)
    dentro = np.ones(A.shape[:3], bool) if mask is None else mask
    info = dict(snr=snr, braccio=braccio, sigma_vera=sig, sigma_mappa_mediana=float(np.median(noise[dentro])),
                lambda_aniso=float(m.lambda_aniso), lambda_iso=float(m.lambda_iso), n_iso=int(m.n_iso),
                catena=catena, opzioni=opzioni)
    t21 = getattr(m, 'two_vs_one_', None)
    if t21:
        info['two_vs_one'] = {k: v for k, v in t21.items() if isinstance(v, (int, float, str, bool)) or v is None}
    return res, m, info


def stima(res, m=None):
    """Mappe stimate col nome breve. Le frazioni sono definite in ogni voxel fittato; il tensore e' NaN
    dove il toolbox non ha trovato fibre (lasciato NaN: e' "fibra mancata", contato a parte)."""
    out = {k: res[..., IX[c]].astype(float) for k, c in METRICHE.items()}
    out['npop'] = np.nan_to_num(res[..., IX['n_fiber_populations']].astype(float))
    if m is not None and getattr(m, 'uncertainty_', None) is not None:
        se = m.uncertainty_['se']
        for k, c in METRICHE.items():
            out['SE_' + k] = se[..., IX[c]].astype(float)
    return out


# ─────────────────────────────────────────────────────────────────────────────
# misure
# ─────────────────────────────────────────────────────────────────────────────
def interno(lab, margine=1):
    """Maschera dei voxel ad almeno `margine` voxel dal bordo del proprio blocco (lungo x e y): ai
    bordi dwidenoise mescola blocchi diversi, come al confine fra tessuti."""
    ok = lab >= 0
    for ax in (0, 1):
        for s in range(1, margine + 1):
            ok &= np.roll(lab, s, ax) == lab
            ok &= np.roll(lab, -s, ax) == lab
    return ok


def tabella_errori(est, tru, gruppi, nomi_gruppi, sel=None):
    """Per gruppo e metrica: vero (mediana), stima (mediana), bias (media), MAE, sd, n validi, e per le
    metriche del tensore la quota di voxel con fibra vera in cui la stima manca (NaN)."""
    righe = []
    sel = np.ones(gruppi.shape, bool) if sel is None else sel
    for g, nome in enumerate(nomi_gruppi):
        s = (gruppi == g) & sel
        for k in METRICHE:
            t, e = tru[k][s], est[k][s]
            ok = np.isfinite(t) & np.isfinite(e)
            r = dict(gruppo=nome, metrica=k, n=int(s.sum()), n_validi=int(ok.sum()))
            if k in TENSORE:
                vf = np.isfinite(t)
                r['mancanti_pct'] = float(100 * np.mean(~np.isfinite(e[vf]))) if vf.any() else np.nan
            if ok.sum():
                d = (e[ok] - t[ok]) * SCALA[k]
                r.update(vero=float(np.median(t[ok]) * SCALA[k]), stima=float(np.median(e[ok]) * SCALA[k]),
                         bias=float(d.mean()), mae=float(np.abs(d).mean()), sd=float(d.std()))
            righe.append(r)
        tn, en = tru['npop'][s], est['npop'][s]
        righe.append(dict(gruppo=nome, metrica='npop', n=int(s.sum()), n_validi=int(s.sum()),
                          vero=float(np.median(tn)), stima=float(np.mean(en)),
                          giusto_pct=float(100 * np.mean(np.round(en) == np.round(tn)))))
    return pd.DataFrame(righe)


def errore_angolare(res, dirs_vere):
    """Errore angolare (gradi) di ogni popolazione stimata rispetto alla fibra vera piu' vicina.
    dirs_vere: (..., 2, 3) con NaN dove la fibra non c'e'. Ritorna (..., 2), NaN se manca una delle due."""
    out = np.full(res.shape[:-1] + (2,), np.nan)
    V = np.nan_to_num(dirs_vere)
    for p in (0, 1):
        e = np.stack([res[..., IX[f'dir{p + 1}_{c}']] for c in 'xyz'], -1).astype(float)
        cos = np.abs(np.einsum('...c,...kc->...k', np.nan_to_num(e), V))        # (..., 2)
        cos[~np.isfinite(dirs_vere[..., 0])] = -1
        best = np.clip(cos.max(-1), 0, 1)
        ok = np.isfinite(e).all(-1) & np.isfinite(dirs_vere[..., 0, 0])
        out[..., p] = np.where(ok, np.degrees(np.arccos(best)), np.nan)
    return out


def versione():
    return dbsi_toolbox.__version__

In [ ]:
"""
La matematica di pyDBSI in figure: una funzione per ogni sezione del voxel journey.

Regola: STRUMENTARE, NON RIPRODURRE. Dove possibile le figure usano le funzioni vere del toolbox e i
dati che il fit usa davvero: `intercetta_fit` lancia `DBSI_Adaptive.fit()` e registra gli argomenti del
kernel (segnale corretto, A^T A regolarizzata, dizionario, grafo dei vicini, lambda), e `registra_calibrazione`
registra ingressi e uscite di ogni passo della calibrazione. Dove una figura deve mostrare un passaggio
interno che il toolbox non restituisce (i bacini, i candidati del cono, la traiettoria dell'LM), lo ricalcola
con le stesse funzioni e CONTROLLA che il risultato coincida con quello del toolbox.
"""

import contextlib
import inspect
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

import dbsi_toolbox
from dbsi_toolbox import DBSI_Adaptive
import dbsi_toolbox.model_Niso_adaptive_ff_thr as M
import dbsi_toolbox.calibration.data_driven as DD
from dbsi_toolbox.core import solvers as SOL
from dbsi_toolbox.core.basis import build_design_matrix_exhaustive, generate_anchored_isotropic_grid

CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
SEQ = LinearSegmentedColormap.from_list('seq_blu', ['#f7f9fc', '#cde2fb', '#86b6ef', '#3987e5',
                                                    '#256abf', '#184f95', '#0d366b'])
COMP = dict(R=CAT[7], H=CAT[0], W=CAT[2], F=CAT[1])          # restricted, hindered, acqua, fibra
D_NOM = dict(R=0.15e-3, H=1.0e-3, W=3.0e-3)
NOMI = DBSI_Adaptive.output_map_names(3)
IX = {n: i for i, n in enumerate(NOMI)}


def stile():
    plt.rcParams.update({
        'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight', 'font.size': 9,
        'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'axes.titlesize': 10, 'axes.titleweight': 'bold',
        'xtick.color': INK2, 'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False,
        'grid.color': GRID, 'grid.linewidth': 0.6, 'legend.frameon': False,
        'figure.facecolor': 'white', 'axes.facecolor': 'white'})


def _salva(fig, file):
    if file:
        fig.savefig(file)
    return fig


# ═════════════════════════════════════════════════════════════════════════════
# strumenti: intercettare il fit e la calibrazione
# ═════════════════════════════════════════════════════════════════════════════
def intercetta_fit(prot, data, mask, noise_map, lam=None, opzioni=None, **kw_fit):
    """Fit vero; registra gli argomenti del kernel 3-ISO. lam=(lambda_aniso, lambda_iso) per congelarli
    (niente calibrazione); opzioni = altri argomenti di DBSI_Adaptive (es. quelle dei crossing). Ritorna dict
    con gli argomenti per nome + 'res' + 'model'."""
    rec = {}
    orig = M._fit_voxels_3iso_v3
    nomi = list(inspect.signature(orig.py_func).parameters)

    def spia(*a, **k):
        rec.update(dict(zip(nomi, a)))
        rec.update(k)
        return orig(*a, **k)

    kwm = {} if lam is None else dict(lambda_aniso=lam[0], lambda_iso=lam[1], n_iso=6)
    kwm.update(opzioni or {})
    m = DBSI_Adaptive(**kwm)
    M._fit_voxels_3iso_v3 = spia
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            res, mode = m.fit(data, prot[0], prot[1], mask, run_calibration=lam is None,
                              noise_map=noise_map, **kw_fit)
    finally:
        M._fit_voxels_3iso_v3 = orig
    assert mode == 3 and 'AtA_reg' in rec, 'kernel 3-ISO non intercettato'
    rec.update(res=res, model=m, prot=prot, n_pairs=len(rec['diff_pairs']),
               n_aniso=len(rec['diff_pairs']) * int(rec['n_dirs']))
    At = np.asarray(rec['At'])
    rec['AtA'] = At @ At.T       # non regolarizzata (A^T A)
    return rec


def registra_calibrazione(prot, data, mask, noise_map, n_cal=1000, seed=None):
    """Calibrazione vera (_calibration_only) con un registratore su ogni passo. seed: se dato, sostituisce
    il seme fisso 0 del campionamento (per studiare la variabilita' del campione)."""
    R = dict(nnls_lambda=[])
    orig = {n: getattr(DD, n) for n in ('select_lambda_iso_gcv', 'select_lambda_aniso_discrepancy',
                                        'lambda_iso_discrepancy_cap', 'select_lambda_aniso_gcv_lcurve',
                                        'nnls_batch')}
    orig_samp = M.sample_calibration_voxels

    def gcv_iso(bvals, iso_grid, y, lambda_grid=None):
        out = orig['select_lambda_iso_gcv'](bvals, iso_grid, y, lambda_grid=lambda_grid)
        R.setdefault('gcv_iso', []).append(dict(y=np.array(y), iso_grid=np.array(iso_grid), lam=out[0], diag=out[1]))
        return out

    def disc(AtA, At, y, n_aniso_cols, sigma, lambda_iso_fixed, **k):
        R['_AtA00'] = float(AtA[0, 0])       # il lambda valutato si legge dalla diagonale di AtA_reg
        R['disc_in'] = dict(AtA=AtA, At=At, y=np.array(y), n_aniso_cols=n_aniso_cols, sigma=sigma,
                            lambda_iso=lambda_iso_fixed, k=k)
        R['nnls_lambda'] = []
        out = orig['select_lambda_aniso_discrepancy'](AtA, At, y, n_aniso_cols, sigma, lambda_iso_fixed, **k)
        R['disc_out'] = out
        R['disc_lambda_valutati'] = list(R['nnls_lambda'])
        return out

    def cap(bvals, iso_grid, y_iso_resid, sigma, lambda_grid):
        out = orig['lambda_iso_discrepancy_cap'](bvals, iso_grid, y_iso_resid, sigma, lambda_grid)
        R['cap'] = dict(y=np.array(y_iso_resid), sigma=sigma, grid=np.array(lambda_grid), out=out, iso_grid=iso_grid)
        return out

    def gcv_aniso(AtA, At, y, n_aniso_cols, lambda_iso_fixed, method='gcv', lambda_grid=None):
        out = orig['select_lambda_aniso_gcv_lcurve'](AtA, At, y, n_aniso_cols, lambda_iso_fixed,
                                                     method=method, lambda_grid=lambda_grid)
        R['aniso'] = dict(lam=out[0], diag=out[1], N=np.asarray(y).shape[1])
        return out

    def batch(AtA_reg, Aty_rows, reg):
        R['nnls_lambda'].append(float(AtA_reg[0, 0] - R['_AtA00']) if '_AtA00' in R else np.nan)
        return orig['nnls_batch'](AtA_reg, Aty_rows, reg)

    def samp(*a, **k):
        if seed is not None:
            k['seed'] = seed
        out = orig_samp(*a, **k)
        R['campione'] = dict(y=np.array(out[0]), sigma_cal=out[1])
        return out

    DD.select_lambda_iso_gcv, DD.select_lambda_aniso_discrepancy = gcv_iso, disc
    DD.lambda_iso_discrepancy_cap, DD.select_lambda_aniso_gcv_lcurve = cap, gcv_aniso
    DD.nnls_batch = batch
    M.sample_calibration_voxels = samp
    try:
        m = DBSI_Adaptive()
        with contextlib.redirect_stdout(io.StringIO()):
            m.fit(data, prot[0], prot[1], mask, run_calibration=True, n_calibration_voxels=n_cal,
                  noise_map=noise_map, _calibration_only=True)
    finally:
        for n, f in orig.items():
            setattr(DD, n, f)
        M.sample_calibration_voxels = orig_samp
    R.update(lambda_aniso=float(m.lambda_aniso), lambda_iso=float(m.lambda_iso), model=m)
    return R


# ═════════════════════════════════════════════════════════════════════════════
# 3.3  protocollo: 3-ISO o 2-ISO
# ═════════════════════════════════════════════════════════════════════════════
def fig_protocollo(prot, snr=26, protocolli=None, file=None):
    bv = prot[0]
    b = np.linspace(0, 3000, 400)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    a = axs[0]
    for k, lab in (('R', 'restricted  D = 0.15'), ('H', 'hindered  D = 1.0'), ('W', 'acqua libera  D = 3.0')):
        a.plot(b, np.exp(-b * D_NOM[k]), color=COMP[k], lw=2, label=lab + ' ×1e-3 mm²/s')
    a.fill_between(b, np.exp(-b * 0.7e-3), np.exp(-b * 1.3e-3), color=COMP['H'], alpha=0.12, lw=0,
                   label='hindered reale (0.7–1.3)')
    a.axhspan(0, 1 / snr, color='#9a988f', alpha=0.25, lw=0)
    a.text(60, 1.15 / snr, f'livello del rumore  σ/S0 = 1/{snr}', color=INK2, fontsize=8)
    gusci = pd.Series(np.round(bv, -2).astype(int)).value_counts().sort_index()
    for g, n in gusci.items():
        if g > 50:
            a.axvline(g, color=INK2, lw=0.8, ls=':')
            a.text(g, 1.02, f'b={g}\n{n} dir', ha='center', va='bottom', fontsize=7, color=INK2)
    for k in 'HW':
        v = np.exp(-2000 * D_NOM[k])
        a.plot(2000, v, 'o', color=COMP[k], ms=6, mec='white')
        a.annotate(f'{v:.3f}', (2000, v), xytext=(8, 4 if k == 'H' else -10), textcoords='offset points',
                   fontsize=8, color=COMP[k])
    a.set_xlabel('b [s/mm²]'); a.set_ylabel('segnale S/S0'); a.set_ylim(0, 1.12)
    a.set_title('Ogni compartimento decade con la sua D')
    a.legend(loc='center right', bbox_to_anchor=(1.0, 0.6), fontsize=7.5)
    a = axs[1]
    contr = np.exp(-b * D_NOM['H']) - np.exp(-b * D_NOM['W'])
    a.plot(b, contr, color=INK, lw=2, label='hindered − acqua')
    a.axhline(2 / snr, color='#9a988f', ls='--', lw=1.2, label=f'2σ (SNR {snr})')
    a.fill_between(b, 0, contr, where=b >= 1500, color=CAT[2], alpha=0.15, lw=0)
    a.set_xlabel('b [s/mm²]'); a.set_ylabel('differenza di segnale')
    a.set_title('Separare hindered e acqua: serve b alto')
    a.text(1550, 0.03, "a b alto l'acqua è già sparita,\nla hindered no: le due curve\nsi separano", fontsize=8,
           color=INK2)
    a.legend(loc='upper right', fontsize=8)
    if protocolli:
        righe = []
        for nome, p in protocolli.items():
            g = sorted(set(np.round(p[0], -2).astype(int)) - {0})
            bmax, nsh, tre, _ = M.analyse_protocol(p[0])
            righe.append(f"{nome}: b_max {bmax:.0f}, {nsh} gusci -> {'3-ISO' if tre else '2-ISO'}")
        fig.text(0.01, -0.06, '   '.join(righe), fontsize=8, color=INK2)
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 4  rumore Rician
# ═════════════════════════════════════════════════════════════════════════════
def _rice_media(A, s):
    from scipy.special import i0e, i1e
    x = A ** 2 / (4 * s ** 2)
    return s * np.sqrt(np.pi / 2) * ((1 + 2 * x) * i0e(x) + 2 * x * i1e(x))


def fig_rician(snr=26, prot=None, file=None, seed=0):
    from scipy.stats import rice
    rng = np.random.default_rng(seed)
    s = 1.0
    fig, axs = plt.subplots(1, 3, figsize=(13.5, 3.9), constrained_layout=True)
    a = axs[0]
    m = np.linspace(0, 8, 400)
    for i, A in enumerate((0, 1, 2, 4)):
        a.plot(m, rice.pdf(m, A / s, scale=s), color=CAT[i], lw=2, label=f'A = {A}σ')
        a.axvline(A, color=CAT[i], lw=0.8, ls=':')
    a.set_xlabel('magnitudine misurata M  [σ]'); a.set_ylabel('densità')
    a.set_title('M = |A + n₁ + i n₂| segue Rice')
    a.legend(fontsize=8)
    a.text(4.6, 0.45, 'a segnale basso la\ndistribuzione è spostata\na destra: M > A in media', fontsize=8,
           color=INK2)
    a = axs[1]
    A = np.linspace(0, 5, 200)
    em = _rice_media(A, s)
    a.plot(A, A, color=INK, lw=1, ls='--', label='vero A')
    a.plot(A, em, color=CAT[7], lw=2, label='E[M] (dato grezzo)')
    a.axhline(np.sqrt(np.pi / 2), color=CAT[7], lw=0.8, ls=':')
    a.text(3.2, np.sqrt(np.pi / 2) + 0.08, 'pavimento σ√(π/2) ≈ 1.25σ', color=CAT[7], fontsize=8)
    # correzione per misura: sqrt(max(M^2 - 2 sigma^2, 0)), media su molte realizzazioni
    corr = []
    for a_ in A:
        Mm = np.sqrt((a_ + rng.normal(0, s, 20000)) ** 2 + rng.normal(0, s, 20000) ** 2)
        c = np.sqrt(np.maximum(Mm ** 2 - 2 * s ** 2, 0))
        corr.append((c.mean(), np.sqrt(np.mean(c ** 2))))
    corr = np.array(corr)
    a.plot(A, corr[:, 1], color=CAT[2], lw=2, label='√E[max(M²−2σ², 0)] (corretto)')
    a.plot(A, corr[:, 0], color=CAT[2], lw=1.2, ls='-.', label='E[√max(M²−2σ², 0)]')
    a.set_xlabel('segnale vero A  [σ]'); a.set_ylabel('[σ]')
    a.set_title('E[M²] = A² + 2σ²: la correzione toglie 2σ²')
    a.legend(fontsize=7.5, loc='upper left')
    a = axs[2]
    if prot is None:
        bv = np.repeat([0, 300, 700, 1000, 2000], [9, 3, 12, 20, 48]).astype(float)
    else:
        bv = prot[0]
    S0, sg = 1000.0, 1000.0 / snr
    vero = S0 * (0.95 * np.exp(-bv * 3.0e-3) + 0.05 * np.exp(-bv * 1.0e-3))      # liquor
    Mm = np.sqrt((vero[None] + rng.normal(0, sg, (400, bv.size))) ** 2 + rng.normal(0, sg, (400, bv.size)) ** 2)
    C = np.sqrt(np.maximum(Mm ** 2 - 2 * sg ** 2, 0))
    o = np.argsort(bv, kind='stable')
    bb = np.unique(bv)
    mean_raw = [Mm[:, bv == x].mean() / S0 for x in bb]
    mean_cor = [C[:, bv == x].mean() / S0 for x in bb]
    vv = [vero[bv == x].mean() / S0 for x in bb]
    a.semilogy(bb, vv, 'o-', color=INK, lw=1.5, label='vero (liquor: 95% acqua)')
    a.semilogy(bb, mean_raw, 's-', color=CAT[7], lw=1.5, label='grezzo: si appoggia al pavimento')
    a.semilogy(bb, mean_cor, '^-', color=CAT[2], lw=1.5, label='corretto')
    a.axhline(sg * np.sqrt(np.pi / 2) / S0, color=CAT[7], lw=0.8, ls=':')
    # cosa direbbe Stage D (NNLS sui tre centroidi) del segnale grezzo e corretto
    rf = []
    for sig in (np.array(mean_raw), np.array(mean_cor)):
        Aiso = np.exp(-np.outer(bb, [D_NOM['R'], D_NOM['H'], D_NOM['W']]))
        from scipy.optimize import nnls
        w, _ = nnls(Aiso, sig)
        rf.append(w / w.sum())
    a.text(0.02, 0.04, f'NNLS su R/H/W (come Stage D):\n  grezzo     RF {rf[0][0]:.2f}  WF {rf[0][2]:.2f}\n'
           f'  corretto   RF {rf[1][0]:.2f}  WF {rf[1][2]:.2f}\n  vero       RF 0.00  WF 0.95',
           transform=a.transAxes, fontsize=8, family='monospace', color=INK)
    a.set_xlabel('b [s/mm²]'); a.set_ylabel('S/S0 (scala log)')
    a.set_title(f'Senza correzione il pavimento sembra RF (SNR {snr})')
    a.legend(fontsize=7.5, loc='upper right')
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 5.3  griglia isotropa ancorata e matrice A
# ═════════════════════════════════════════════════════════════════════════════
def fig_griglia(file=None):
    d_min, d_max, n = 0.1e-3, max(3.0e-3, M._ISO_GRID_D_MAX_EXTENDED), M._QUADRATURE_N_ISO
    tr, tw, eps = M.THRESH_RES, M.THRESH_WAT, 1e-6
    p1 = np.geomspace(d_min, d_max, n)
    finale = np.asarray(generate_anchored_isotropic_grid(d_min=d_min, d_max=d_max, n_steps=n,
                                                        thresh_res=tr, thresh_wat=tw))
    anc = np.array([tr - eps * 1e-3, tw + eps * 1e-3])
    p2 = np.sort(np.concatenate([p1, anc]))
    ins = np.array([x for x in finale if not np.any(np.isclose(x, p2, rtol=1e-6))])
    fig, ax = plt.subplots(figsize=(11, 3.6), constrained_layout=True)
    righe = [('1. 6 punti log-uniformi su [0.1, 5.0]', p1, CAT[0]),
             ('2. + 2 punti ancorati alle soglie', p2, CAT[1]),
             (f'3. + medie geometriche dove D(j+1)/D(j) > 2  →  {len(finale)} colonne', finale, INK)]
    for r, (lab, pts, col) in enumerate(righe):
        y = 2 - r
        ax.plot(pts * 1e3, np.full(len(pts), y), 'o', color=col, ms=7, mec='white', zorder=3)
        ax.text(0.085, y + 0.22, lab, fontsize=8.5, color=INK)
        if r == 1:
            ax.plot(anc * 1e3, [y, y], 'o', color=CAT[1], ms=11, mfc='none', mew=2)
        if r == 2 and len(ins):
            ax.plot(ins * 1e3, np.full(len(ins), y), 'o', color=CAT[2], ms=11, mfc='none', mew=2)
    for t, lab in ((tr, 'soglia R | H  0.3'), (tw, 'soglia H | W  3.0')):
        ax.axvline(t * 1e3, color=INK2, ls='--', lw=1)
        ax.text(t * 1e3 * 1.03, 2.55, lab, fontsize=8, color=INK2)
    for (lo, hi, k, lab) in ((0.08, tr * 1e3, 'R', 'restricted'), (tr * 1e3, tw * 1e3, 'H', 'hindered'),
                             (tw * 1e3, 6.0, 'W', 'acqua libera')):
        ax.axvspan(lo, hi, color=COMP[k], alpha=0.07, lw=0)
        ax.text(np.sqrt(lo * hi), -0.7, lab, ha='center', fontsize=8.5, color=COMP[k], fontweight='bold')
    ax.set_xscale('log'); ax.set_xlim(0.08, 6.0); ax.set_ylim(-0.9, 2.8)
    ax.set_yticks([]); ax.spines['left'].set_visible(False)
    ax.set_xlabel('D [×1e-3 mm²/s]  (scala logaritmica)')
    nr = int((finale <= tr).sum()); nw = int((finale > tw).sum())
    ax.set_title(f'Griglia isotropa ancorata: {len(finale)} colonne ({nr} R, {len(finale) - nr - nw} H, {nw} W)')
    return _salva(fig, file), finale


def fig_matrice_A(K, file=None):
    """La matrice di Stage A del protocollo: righe = misure ordinate per b, colonne = blocco anisotropo
    (per coppia (AD, RD), poi per direzione) + blocco isotropo."""
    bv, bc = K['prot']
    A = build_design_matrix_exhaustive(bv, bc, K['fiber_dirs'], K['diff_pairs'], K['iso_grid'])
    o = np.argsort(bv, kind='stable')
    A = A[o]
    na, npair, nd = K['n_aniso'], K['n_pairs'], int(K['n_dirs'])
    fig = plt.figure(figsize=(13, 5.6), constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[3.4, 1.6])
    a = fig.add_subplot(gs[0])
    im = a.imshow(A, aspect='auto', cmap=SEQ, vmin=0, vmax=1, interpolation='nearest')
    for p in range(1, npair + 1):
        a.axvline(p * nd - 0.5, color='white', lw=1.2)
    pos = [p * nd + nd / 2 for p in range(npair)] + [na + len(K['iso_grid']) / 2]
    lab = [f'AD {ad*1e3:.2f}\nRD {rd*1e3:.2f}' for ad, rd in K['diff_pairs']] + ['iso']
    top = a.secondary_xaxis('top')
    top.set_xticks(pos); top.set_xticklabels(lab, fontsize=7)
    bs = bv[o]
    for g in np.unique(np.round(bs, -2)):
        i0 = np.where(np.round(bs, -2) == g)[0][0]
        a.axhline(i0 - 0.5, color='white', lw=0.6)
        a.text(-3, i0, f'b={int(g)}', ha='right', va='top', fontsize=7)
    a.set_yticks([]); a.set_xticks([])
    a.set_xlabel(f'{na} colonne anisotrope ({npair} coppie × {nd} direzioni)  +  {len(K["iso_grid"])} isotrope')
    a.set_title(f'Matrice di Stage A del protocollo: {A.shape[0]} misure × {A.shape[1]} colonne', pad=30)
    fig.colorbar(im, ax=a, shrink=0.8, label='exp(−b D)')
    a2 = fig.add_subplot(gs[1])
    j_dir = 0
    for k, (col, lab, c) in enumerate(((0 * nd + j_dir, 'fibra, coppia 1, direzione 1', CAT[1]),
                                        ((npair - 1) * nd + j_dir, f'fibra, coppia {npair}, direzione 1', CAT[4]),
                                        (na + 2, f'iso D={K["iso_grid"][2]*1e3:.2f}', COMP['R']),
                                        (na + len(K['iso_grid']) - 1, f'iso D={K["iso_grid"][-1]*1e3:.1f}', COMP['W']))):
        a2.plot(A[:, col], color=c, lw=1.2, label=lab)
    a2.set_xlabel('misura (ordinate per b)'); a2.set_ylabel('valore della colonna')
    a2.set_title('Quattro colonne lette per intero')
    a2.legend(fontsize=7)
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 5.5  grafo dei vicini assiale e cono
# ═════════════════════════════════════════════════════════════════════════════
def _proietta(v):
    """Proiezione azimutale equivalente dall'emisfero z >= 0 (antipodi riportati su)."""
    v = np.atleast_2d(v).astype(float).copy()
    v[v[:, 2] < 0] *= -1
    r = np.sqrt(2 * (1 - np.clip(v[:, 2], -1, 1)))
    phi = np.arctan2(v[:, 1], v[:, 0])
    return r * np.cos(phi), r * np.sin(phi)


def _cerchio_equatore(a):
    t = np.linspace(0, 2 * np.pi, 200)
    a.plot(np.sqrt(2) * np.cos(t), np.sqrt(2) * np.sin(t), color=INK2, lw=0.8)
    a.set_aspect('equal'); a.axis('off')


def fig_grafo_e_cono(K, file=None):
    fd = np.asarray(K['fiber_dirs']); nb = np.asarray(K['neighbor_idx'])
    nd, k = nb.shape
    x, y = _proietta(fd)
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.8), constrained_layout=True)
    a = axs[0]
    _cerchio_equatore(a)
    for d in range(nd):
        for j in nb[d]:
            xa, ya = x[d], y[d]
            xb, yb = x[j], y[j]
            if np.hypot(xa - xb, ya - yb) > 1.2:          # vicino oltre l'equatore: e' -v
                continue
            a.plot([xa, xb], [ya, yb], color='#c9c7c0', lw=0.6, zorder=1)
    a.scatter(x, y, s=14, color=CAT[0], zorder=2)
    a.set_title(f'{nd} direzioni, grafo dei vicini (k={k})')
    # una direzione vicina all'equatore: vicini assiali contro prodotto scalare semplice
    eq = int(np.argmin(np.abs(fd[:, 2])))
    a = axs[1]
    _cerchio_equatore(a)
    a.scatter(x, y, s=10, color='#c9c7c0')
    semplice = np.argsort(-(fd @ fd[eq]))[1:k + 1]
    a.scatter(x[semplice], y[semplice], s=70, facecolors='none', edgecolors=CAT[7], lw=1.6,
              label='vicini col prodotto scalare semplice')
    a.scatter(x[nb[eq]], y[nb[eq]], s=28, color=CAT[2], label='vicini assiali |v·v\'| (toolbox)')
    a.scatter(x[eq], y[eq], s=60, color=INK, marker='*', label='direzione vicina all\'equatore')
    a.legend(fontsize=7.5, loc='lower center', bbox_to_anchor=(0.5, -0.18))
    a.set_title('All\'equatore v e −v sono la stessa fibra')
    # schema del cono
    sp = SOL.measure_hemisphere_spacing(fd)
    a1, n1, a2, n2 = SOL.compute_cone_refinement_schedule(sp, np.radians(1.0))
    a = axs[2]
    v0 = fd[int(np.argmax(fd[:, 2]))]
    ex, ey = _base(v0)
    c1 = np.array([SOL._fibonacci_cone_point(*v0, *ex, *ey, a1, i, int(n1)) for i in range(int(n1))])
    best = c1[int(n1) // 3]
    bx_, by_ = _base(best)
    c2 = np.array([SOL._fibonacci_cone_point(*best, *bx_, *by_, a2, i, int(n2)) for i in range(int(n2))])
    def loc(v):            # coordinate locali in gradi attorno a v0
        return (np.degrees(np.arctan2(v @ ex, v @ v0)), np.degrees(np.arctan2(v @ ey, v @ v0)))
    vic = fd[np.argsort(-np.abs(fd @ v0))[:7]]
    lx, ly = loc(vic * np.sign(vic @ v0)[:, None])
    a.scatter(lx, ly, s=60, color='#c9c7c0', label='nodi del dizionario')
    p1 = loc(c1); p2 = loc(c2)
    a.scatter(*p1, s=26, color=CAT[0], label=f'livello 1: {int(n1)} candidati, ±{np.degrees(a1):.1f}°')
    a.scatter(*p2, s=12, color=CAT[1], label=f'livello 2: {int(n2)} candidati, ±{np.degrees(a2):.1f}°')
    t = np.linspace(0, 2 * np.pi, 100)
    a.plot(np.degrees(a1) * np.cos(t), np.degrees(a1) * np.sin(t), color=CAT[0], lw=0.8)
    bl = loc(best[None])
    a.plot(bl[0] + np.degrees(a2) * np.cos(t), bl[1] + np.degrees(a2) * np.sin(t), color=CAT[1], lw=0.8)
    a.scatter(0, 0, s=70, color=INK, marker='*', label='nodo di partenza (Stage A)')
    a.set_aspect('equal'); a.set_xlabel('gradi'); a.set_ylabel('gradi')
    a.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.0, 1.0))
    a.set_title(f'Il cono: spaziatura {np.degrees(sp):.1f}° → risoluzione 1°')
    return _salva(fig, file), dict(spaziatura=np.degrees(sp), a1=np.degrees(a1), n1=int(n1), a2=np.degrees(a2), n2=int(n2), k=k)


def _base(v0):
    """La stessa base ortonormale del toolbox (refine_fiber_direction_cone)."""
    t = np.array([1.0, 0, 0]) if abs(v0[0]) < 0.9 else np.array([0, 1.0, 0])
    x = np.cross(v0, t); x /= np.linalg.norm(x)
    y = np.cross(v0, x)
    return x, y


# ═════════════════════════════════════════════════════════════════════════════
# 6.1  convergenza di n_iso (notebook 06, dati veri, v1.3.6)
# ═════════════════════════════════════════════════════════════════════════════
CONVERGENZA_06 = pd.DataFrame({
    'n_iso': [4, 6, 8, 10, 12, 16, 20, 24, 31, 40], 'colonne': [9, 11, 10, 12, 14, 18, 22, 25, 33, 42],
    'FF': [0.5013, 0.4998, 0.4991, 0.4994, 0.4993, 0.5028, 0.5037, 0.5046, 0.5037, 0.5027],
    'RF': [0.1108, 0.1115, 0.1118, 0.1107, 0.1115, 0.1120, 0.1123, 0.1123, 0.1123, 0.1120],
    'HF': [0.4450, 0.4422, 0.4487, 0.4449, 0.4471, 0.4472, 0.4443, 0.4414, 0.4397, 0.4404],
    'WF': [0.1713, 0.1718, 0.1718, 0.1712, 0.1715, 0.1716, 0.1710, 0.1708, 0.1709, 0.1707],
    'FA': [0.6759, 0.6842, 0.6757, 0.6841, 0.6850, 0.6889, 0.6893, 0.6895, 0.6876, 0.6893],
    'NNLS_non_conv_pct': [0.0, 0.0, 0.0, 0.03, 0.77, 3.7, 7.0, 10.0, 19.5, 40.2],
    'secondi': [126, 121, 140, 133, 153, 203, 254, 303, 409, 614]})


def fig_convergenza(sintetica=None, file=None):
    T = CONVERGENZA_06
    fig, axs = plt.subplots(1, 3, figsize=(13.5, 3.8), constrained_layout=True)
    a = axs[0]
    for i, k in enumerate(('FF', 'RF', 'HF', 'WF')):
        a.plot(T.n_iso, T[k], 'o-', color=[COMP['F'], COMP['R'], COMP['H'], COMP['W']][i], lw=1.8, label=k)
    a.axvline(6, color=INK2, ls='--', lw=1); a.text(6.3, 0.52, 'n_iso = 6', fontsize=8, color=INK2)
    a.set_xlabel('n_iso'); a.set_ylabel('mediana su 3000 voxel')
    a.set_title('Frazioni: non si muovono (Stage D)')
    a.legend(fontsize=8, ncol=4, loc='center right')
    a = axs[1]
    a.plot(T.n_iso, T.FA, 'o-', color=INK, lw=1.8)
    rif = T[T.n_iso.isin([16, 20])].FA.mean()
    a.axhspan(rif - 0.01, rif + 0.01, color=CAT[2], alpha=0.15, lw=0, label='riferimento ±0.01 (media 16, 20)')
    a.axvline(6, color=INK2, ls='--', lw=1)
    a.set_xlabel('n_iso'); a.set_ylabel('FA mediana')
    a.set_title('Tensore: dentro la banda da 6')
    a.legend(fontsize=7.5, loc='lower right')
    a = axs[2]
    a.bar(T.n_iso.astype(str), T.NNLS_non_conv_pct, color=CAT[7])
    a2 = a.twinx()
    a2.plot(T.n_iso.astype(str), T.secondi, 'o-', color=INK2, lw=1)
    a2.set_ylabel('secondi (3000 voxel)', color=INK2)
    a.set_xlabel('n_iso'); a.set_ylabel('NNLS non convergenti [%]', color=CAT[7])
    a.set_title('Più colonne = più costo, niente informazione')
    fig.suptitle('Convergenza di n_iso — notebook 06, un soggetto vero, 3000 voxel (v1.3.6)', fontsize=10)
    if sintetica is not None:
        fig2, b = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
        for i, k in enumerate(('FF', 'RF', 'HF', 'WF')):
            b[0].plot(sintetica.n_iso, sintetica[k], 'o-', color=[COMP['F'], COMP['R'], COMP['H'], COMP['W']][i], label=k)
        b[0].set_xlabel('n_iso'); b[0].set_ylabel('MAE contro la verità'); b[0].legend(fontsize=8, ncol=4)
        b[0].set_title(f'Fantoccio, toolbox {dbsi_toolbox.__version__}: errore delle frazioni')
        b[1].plot(sintetica.n_iso, sintetica['RDw'], 'o-', color=INK, label='RD pesata')
        b[1].set_xlabel('n_iso'); b[1].set_ylabel('MAE [e-3 mm²/s]'); b[1].legend(fontsize=8)
        b[1].set_title('ed errore del tensore')
        if file:
            fig2.savefig(str(file).replace('.png', '_sintetica.png'))
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 7  il campione di calibrazione e il suo sigma
# ═════════════════════════════════════════════════════════════════════════════
def rapporti_sigma(data_corr, mask, bvals, noise_map, b0_thr=100.0):
    """sigma_v / S0_v su tutta la maschera, con la stessa esclusione del toolbox (S0 < 5% della mediana)."""
    b0 = bvals < b0_thr
    S0 = data_corr[..., b0].mean(-1)[mask]
    ok = S0 > 0.05 * np.median(S0)
    return (noise_map[mask][ok] / S0[ok]).astype(float)


def fig_sigma_campione(r, n_list=(100, 250, 500, 1000, 2000, 4000), n_rip=200, file=None, seed=0):
    rng = np.random.default_rng(seed)
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.9), constrained_layout=True)
    a = axs[0]
    hi = np.percentile(r, 99.5)
    a.hist(np.clip(r, 0, hi), bins=80, color=CAT[0], alpha=0.75)
    med, rms = np.median(r), np.sqrt(np.mean(r ** 2))
    a.axvline(med, color=CAT[1], lw=2, label=f'mediana {med:.4f}')
    a.axvline(rms, color=CAT[2], lw=2, label=f'RMS {rms:.4f}')
    a.set_xlabel('σ_v / S0_v  (rumore normalizzato del voxel)'); a.set_ylabel('voxel')
    a.set_title(f'Eterogeneo: RMS² / mediana² = {(rms / med) ** 2:.2f}')
    a.legend(fontsize=8)
    a = axs[1]
    out = []
    for n in n_list:
        for _ in range(n_rip):
            s = rng.choice(r, size=min(n, len(r)), replace=False)
            out.append(dict(n=n, mediana=np.median(s) / med - 1, RMS=np.sqrt(np.mean(s ** 2)) / rms - 1))
    D = pd.DataFrame(out)
    pos = np.arange(len(n_list))
    for j, (k, c) in enumerate((('mediana', CAT[1]), ('RMS', CAT[2]))):
        q = D.groupby('n')[k].quantile([0.05, 0.5, 0.95]).unstack()
        a.fill_between(pos, 100 * q[0.05], 100 * q[0.95], color=c, alpha=0.2, lw=0)
        a.plot(pos, 100 * q[0.5], 'o-', color=c, lw=1.8, label=f'{k}: 5–95° percentile')
    a.axhline(0, color=INK2, lw=0.8)
    a.set_xticks(pos); a.set_xticklabels([str(n) for n in n_list])
    a.set_xlabel('voxel nel campione'); a.set_ylabel('scarto dal valore su tutto il cervello [%]')
    a.set_title('Quanto cambia σ_cal fra campioni diversi')
    a.legend(fontsize=8)
    return _salva(fig, file), D


def fig_lambda_vs_campione(tab, file=None):
    """tab: colonne n, seme, lambda_aniso, lambda_iso."""
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
    ns = sorted(tab.n.unique())
    for a, k in zip(axs, ('lambda_aniso', 'lambda_iso')):
        for i, n in enumerate(ns):
            v = tab[tab.n == n][k]
            a.scatter(np.full(len(v), i) + np.linspace(-0.12, 0.12, len(v)), v, color=CAT[0], s=24, zorder=3)
            vals, cnt = np.unique(np.round(v, 6), return_counts=True)
            a.text(i, v.max() * 1.15, f'{cnt.max()}/{len(v)} uguali', ha='center', fontsize=7.5, color=INK2)
        a.set_yscale('log'); a.set_xticks(range(len(ns))); a.set_xticklabels([str(n) for n in ns])
        a.set_xlabel('voxel di calibrazione'); a.set_ylabel(k)
        a.set_title(f'{k}: un punto per seme del campione')
        a.grid(axis='y', which='both')
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 8-9  la calibrazione passo per passo
# ═════════════════════════════════════════════════════════════════════════════
def residuo_discrepanza(din, lam_grid):
    """R(lambda) del principio di discrepanza (stessa formula del toolbox, NNLS in parallelo)."""
    AtA, At, y = din['AtA'], din['At'], din['y']
    na, li = din['n_aniso_cols'], din['lambda_iso']
    Aty = np.stack([At @ v for v in y])
    out = []
    for lam in lam_grid:
        reg = np.zeros(AtA.shape[0]); reg[:na] = lam; reg[na:] = li
        W, _ = SOL.nnls_batch(AtA + np.diag(reg), Aty, 0.0)
        r = [max(float(w @ (AtA @ w) - 2 * w @ a + v @ v), 0) for w, a, v in zip(W, Aty, y)]
        out.append(np.mean(r))
    return np.array(out)


def residuo_cap(cap):
    """Residuo medio del blocco isotropo sul residuo senza fibra, per ogni lambda della griglia del tetto."""
    from dbsi_toolbox.core.basis import build_isotropic_dictionary
    bv = cap['bvals']
    A = build_isotropic_dictionary(bv, cap['iso_grid'])
    U, s, _ = np.linalg.svd(A, full_matrices=False)
    y = cap['y']; Uty = U.T @ y.T; yn = np.sum(y ** 2, 1)
    out = []
    for lam in cap['grid']:
        f = s ** 2 / (s ** 2 + lam)
        out.append(np.mean(yn - 2 * np.sum((f[:, None]) * Uty ** 2, 0) + np.sum((f[:, None] ** 2) * Uty ** 2, 0)))
    return np.array(out)


def fig_calibrazione(R, prot, file_prefix=None, n_curva=30):
    """Sei pannelli: passo 1 (GCV iso sul segnale pieno), 2 (discrepanza per lambda_aniso preliminare),
    3 (residuo isotropo su due voxel), 4-5 (GCV sul residuo + tetto), 9 (GCV di lambda_aniso con i dof)."""
    bv = prot[0]
    N = len(bv)
    figs = {}
    g1, g4 = R['gcv_iso'][0], R['gcv_iso'][-1]
    sig = R['disc_in']['sigma']
    fig, axs = plt.subplots(2, 3, figsize=(15, 8.2), constrained_layout=True)
    # passo 1
    a = axs[0, 0]
    a.loglog(g1['diag']['lambda_'], g1['diag']['gcv'], color=CAT[0], lw=2)
    a.axvline(g1['lam'], color=CAT[1], ls='--'); a.text(g1['lam'] * 1.2, a.get_ylim()[0] * 1.2, f"λ_iso(1) = {g1['lam']:.3g}", color=CAT[1], fontsize=8)
    a.set_xlabel('λ_iso'); a.set_ylabel('GCV'); a.set_title('Passo 1 — GCV del blocco isotropo, segnale pieno')
    # passo 2
    a = axs[0, 1]
    din = R['disc_in']
    lam_g = np.logspace(-6, 4, n_curva)
    Rl = residuo_discrepanza(din, lam_g)
    target = N * sig ** 2
    a.loglog(lam_g, Rl, color=CAT[0], lw=2, label='R(λ): residuo medio')
    a.axhline(target, color=INK2, ls='--', label=f'bersaglio Nσ² = {target:.3g}')
    ev = [l for l in R.get('disc_lambda_valutati', []) if np.isfinite(l) and l > 0]
    if ev:
        Re = residuo_discrepanza(din, np.array(ev))
        a.scatter(ev, Re, c=np.arange(len(ev)), cmap='Oranges', s=26, zorder=3, edgecolors=INK, lw=0.4,
                  label=f'bisezione: {len(ev)} valutazioni')
    la1 = R['disc_out'][0]
    a.axvline(la1, color=CAT[1], ls='--')
    a.text(la1 * 1.5, target * 0.85, f'λ_aniso(1) = {la1:.3g}', color=CAT[1], fontsize=8)
    fl = R['disc_out'][1]
    a.set_xlabel('λ_aniso'); a.set_ylabel('R(λ)')
    a.set_title('Passo 2 — discrepanza: il residuo = rumore')
    a.legend(fontsize=7.5, loc='upper left')
    if fl.get('floor_applied'):
        a.text(0.02, 0.02, f"pavimento applicato ({fl.get('floor_component')})", transform=a.transAxes, fontsize=7.5)
    # passo 3: residuo isotropo per due voxel del campione
    a = axs[0, 2]
    y = din['y']; r = g4['y']
    o = np.argsort(bv, kind='stable')
    an = np.linalg.norm(y - r, axis=1)
    for j, (v, lab) in enumerate(((int(np.argmax(an)), 'voxel con molta fibra'), (int(np.argmin(an)), 'voxel quasi isotropo'))):
        a.plot(y[v][o], color=CAT[j], lw=1.2, label=f'{lab}: segnale y')
        a.plot(r[v][o], color=CAT[j], lw=1.2, ls='--', label='residuo isotropo r = y − A_aniso ŵ')
    a.set_xlabel('misura (ordinate per b)'); a.set_ylabel('S/S0')
    a.set_title('Passo 3 — togliere la fibra')
    a.legend(fontsize=7)
    # passo 4-5
    a = axs[1, 0]
    a.loglog(g4['diag']['lambda_'], g4['diag']['gcv'], color=CAT[0], lw=2, label='GCV sul residuo isotropo')
    a.axvline(g4['lam'], color=CAT[0], ls='--')
    a.set_xlabel('λ_iso'); a.set_ylabel('GCV')
    a2 = a.twinx()
    cp = dict(R['cap']); cp['bvals'] = bv
    rc = residuo_cap(cp)
    a2.loglog(cp['grid'], rc, color=CAT[2], lw=1.6, label='residuo iso (tetto)')
    a2.axhline(N * sig ** 2, color=CAT[2], ls=':', lw=1)
    a2.axvline(cp['out'], color=CAT[2], ls='--')
    a2.set_ylabel('residuo medio', color=CAT[2])
    fin = R['lambda_iso']
    a.set_title(f"Passi 4–5 — GCV {g4['lam']:.3g}, tetto {cp['out']:.3g} → λ_iso {fin:.3g}")
    a.legend(fontsize=7.5, loc='upper left'); a2.legend(fontsize=7.5, loc='upper center')
    # passo 9
    d = R['aniso']['diag']
    a = axs[1, 1]
    a.loglog(d['lambda_grid'], d['gcv'], color=CAT[0], lw=2)
    a.axvline(R['aniso']['lam'], color=CAT[1], ls='--')
    a.text(R['aniso']['lam'] * 1.3, np.nanmin(d['gcv']) * 1.1, f"λ_aniso = {R['aniso']['lam']:.3g}", color=CAT[1], fontsize=8)
    sp = np.nanmax(d['gcv'][np.isfinite(d['gcv'])]) / np.nanmin(d['gcv'])
    a.set_xlabel('λ_aniso'); a.set_ylabel('GCV')
    a.set_title(f"Passo 9 — GCV di λ_aniso (max/min {sp:.1f}{', DEBOLE' if d.get('corner_weak') else ''})")
    a = axs[1, 2]
    a.semilogx(d['lambda_grid'], d['dof'], color=INK, lw=2, label='gradi di libertà ridge')
    a.axhline(N, color=INK2, ls=':', label=f'N = {N} misure')
    i = int(np.argmin(np.abs(d['lambda_grid'] - R['aniso']['lam'])))
    a.plot(d['lambda_grid'][i], d['dof'][i], 'o', color=CAT[1])
    a.text(d['lambda_grid'][i] * 1.3, d['dof'][i], f"{d['dof'][i]:.1f} dof", color=CAT[1], fontsize=8)
    a.set_xlabel('λ_aniso'); a.set_ylabel('dof'); a.set_title('Quanti parametri "efficaci" restano')
    a.legend(fontsize=7.5)
    figs['calibrazione'] = _salva(fig, f'{file_prefix}_passi.png' if file_prefix else None)
    riepilogo = dict(sigma_cal=sig, lambda_iso_passo1=g1['lam'], lambda_aniso_passo2=la1,
                     lambda_iso_gcv=g4['lam'], lambda_iso_tetto=cp['out'], lambda_iso_finale=fin,
                     lambda_aniso_finale=R['aniso']['lam'], dof_al_minimo=float(d['dof'][i]),
                     dof_min=float(np.nanmin(d['dof'])), dof_max=float(np.nanmax(d['dof'])),
                     gcv_max_su_min=float(sp), N=N, voxel=len(din['y']))
    return figs, riepilogo


# ═════════════════════════════════════════════════════════════════════════════
# Stage A di un voxel: dal segnale ai pesi, ai bacini, alla modulazione
# ═════════════════════════════════════════════════════════════════════════════
def stage_a_voxel(K, ijk):
    """Ripete per un voxel quello che il kernel fa in Stage A, con le stesse matrici (intercettate)."""
    x, y, z = ijk
    bv = K['prot'][0]
    sig = np.asarray(K['data'][x, y, z], float)
    s0 = sig[bv < K['b0_thr']].mean()
    sn = sig / s0
    At, AtA_reg = np.asarray(K['At']), np.asarray(K['AtA_reg'])
    Aty = At @ sn
    w1, it = SOL.nnls_coordinate_descent(AtA_reg, Aty, 0.0)
    na, nd, npair = K['n_aniso'], int(K['n_dirs']), K['n_pairs']
    nb, fd = np.asarray(K['neighbor_idx']), np.asarray(K['fiber_dirs'])
    c1 = SOL.dominant_basin_concentration(w1[:na], nd, npair, nb, fd)
    w2 = M._apply_conc_modulation(w1, AtA_reg, Aty, na, nd, npair, nb, fd, K['lambda_aniso_scalar'],
                                  K['conc_mod_c_lo'], K['conc_mod_c_hi'], K['conc_mod_gain'])
    c2 = SOL.dominant_basin_concentration(w2[:na], nd, npair, nb, fd)
    frac = (K['conc_mod_c_hi'] - c1) / (K['conc_mod_c_hi'] - K['conc_mod_c_lo'])
    g = 1 + (K['conc_mod_gain'] - 1) * min(max(frac, 0), 1) if c1 < K['conc_mod_c_hi'] else 1.0
    return dict(sn=sn, s0=s0, w1=w1, w2=w2, c1=c1, c2=c2, g=g,
                ff1=w1[:na].sum() / w1.sum(), ff2=w2[:na].sum() / w2.sum(),
                Wd1=w1[:na].reshape(npair, nd).sum(0), Wd2=w2[:na].reshape(npair, nd).sum(0))


def bacini(Wd, nb, fd):
    """Picchi e bacini come in dominant_basin_concentration (ricalcolati per disegnarli)."""
    nd, k = nb.shape
    picchi = []
    for d in range(nd):
        if Wd[d] <= 0:
            continue
        if all(not (Wd[j] > Wd[d] or (Wd[j] == Wd[d] and j < d)) for j in nb[d]):
            picchi.append(d)
    lab = np.full(nd, -1)
    if picchi:
        cos = np.abs(fd @ fd[picchi].T)
        lab = np.where(Wd > 0, np.argmax(cos, 1), -1)
    massa = np.array([Wd[lab == q].sum() for q in range(len(picchi))])
    c = massa.max() / Wd.sum() if len(massa) and Wd.sum() > 0 else 0.0
    return picchi, lab, massa, c


def fig_concentrazione(K, voxel, file=None):
    """voxel: lista di (nome, (i,j,k), direzioni vere o None)."""
    fd, nb = np.asarray(K['fiber_dirs']), np.asarray(K['neighbor_idx'])
    x, y = _proietta(fd)
    n = len(voxel)
    fig, axs = plt.subplots(2, n, figsize=(3.3 * n, 6.6), constrained_layout=True)
    righe = []
    for j, (nome, ijk, veri) in enumerate(voxel):
        S = stage_a_voxel(K, ijk)
        for r, (Wd, cc, ff, tit) in enumerate(((S['Wd1'], S['c1'], S['ff1'], 'prima soluzione'),
                                               (S['Wd2'], S['c2'], S['ff2'], 'dopo la modulazione'))):
            a = axs[r, j]
            _cerchio_equatore(a)
            pk, lab, massa, c_ric = bacini(Wd, nb, fd)
            col = np.array(['#d9d7d0'] * len(fd), dtype=object)
            for q in range(len(pk)):
                col[lab == q] = CAT[q % len(CAT)]
            sz = 4 + 260 * Wd / max(Wd.max(), 1e-12)
            a.scatter(x, y, s=sz, c=list(col), alpha=0.85, edgecolors='none')
            if veri is not None:
                vx, vy = _proietta(np.asarray(veri))
                a.scatter(vx, vy, s=90, marker='x', color=INK, lw=2)
            assert abs(c_ric - cc) < 1e-9, 'bacini ricalcolati diversi dal toolbox'
            a.set_title(f'{nome}\n{tit}: c = {cc:.2f}, FF {ff:.2f}' if r == 0 else f'{tit}: c = {cc:.2f}, FF {ff:.2f}',
                        fontsize=8.5)
        righe.append(dict(voxel=nome, c_prima=S['c1'], guadagno=S['g'], FF_prima=S['ff1'], FF_dopo=S['ff2'],
                          c_dopo=S['c2']))
    fig.suptitle('Peso anisotropo per direzione (area = peso; colore = bacino; × = fibra vera)', fontsize=10)
    return _salva(fig, file), pd.DataFrame(righe)


def fig_rampa(K, c_fibra=None, c_isotropo=None, file=None):
    clo, chi, G = K['conc_mod_c_lo'], K['conc_mod_c_hi'], K['conc_mod_gain']
    c = np.linspace(0, 1, 400)
    g = np.where(c >= chi, 1, 1 + (G - 1) * np.clip((chi - c) / (chi - clo), 0, 1))
    fig, a = plt.subplots(figsize=(7.5, 3.6), constrained_layout=True)
    a.plot(c, g, color=INK, lw=2.2, label='guadagno g(c) su λ_aniso')
    a.axvline(clo, color=INK2, ls=':'); a.axvline(chi, color=INK2, ls=':')
    a.text(clo, G + 0.2, f'c_lo {clo}', ha='center', fontsize=8); a.text(chi, G + 0.2, f'c_hi {chi}', ha='center', fontsize=8)
    a.set_xlabel('concentrazione c'); a.set_ylabel('g'); a.set_ylim(0.5, G + 1)
    if c_fibra is not None:
        a2 = a.twinx()
        b = np.linspace(0, 1, 41)
        a2.hist(c_isotropo, bins=b, color=CAT[7], alpha=0.35, density=True, label='voxel senza fibra vera')
        a2.hist(c_fibra, bins=b, color=CAT[0], alpha=0.35, density=True, label='voxel con fibra vera')
        a2.set_yticks([]); a2.legend(fontsize=8, loc='center right')
    a.legend(fontsize=8, loc='upper right')
    a.set_title('La rampa: penalità piena dove il peso è diffuso, nessuna dove è concentrato')
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 17.2  il cono su un voxel vero
# ═════════════════════════════════════════════════════════════════════════════
def _frazioni_stage_a(K, w):
    na = K['n_aniso']; iso = np.asarray(K['iso_grid']); wi = w[na:]
    tot = w.sum()
    res = iso <= M.THRESH_RES
    f_res, f_non = wi[res].sum() / tot, wi[~res].sum() / tot
    D_res = (wi[res] * iso[res]).sum() / max(wi[res].sum(), 1e-12) if wi[res].sum() > 1e-10 else 0.15e-3
    D_non = (wi[~res] * iso[~res]).sum() / max(wi[~res].sum(), 1e-12) if wi[~res].sum() > 1e-10 else 1.5e-3
    return w[:na].sum() / tot, f_res, f_non, D_res, D_non


def fig_cono(K, ijk, vera=None, file=None):
    bv, bc = K['prot']
    S = stage_a_voxel(K, ijk)
    fd, nb = np.asarray(K['fiber_dirs']), np.asarray(K['neighbor_idx'])
    na, nd, npair = K['n_aniso'], int(K['n_dirs']), K['n_pairs']
    idx, _ = SOL.select_dominant_directions(S['w2'][:na], nd, npair, nb, fd, 2, K['min_weight_fraction'],
                                            K['min_separation_cos'], K['min_peak_ratio'],
                                            K['min_dominant_concentration'])
    v0 = fd[idx[0]]                      # il nodo dominante, come nel kernel
    f_fib, f_res, f_non, D_res, D_non = _frazioni_stage_a(K, S['w2'])
    a1, n1, a2, n2 = K['cone1_half_angle'], int(K['n1_cone']), K['cone2_half_angle'], int(K['n2_cone'])

    def sse(d):
        AD, RD = SOL.estimate_AD_RD_conditioned(bv, bc, S['sn'], d, f_fib, f_res, f_non, 0.0, D_res, D_non, 0.0, False)
        if np.isnan(AD):
            return np.nan
        cos = bc @ d
        pred = f_fib * np.exp(-bv * (RD + (AD - RD) * cos ** 2)) + f_res * np.exp(-bv * D_res) + f_non * np.exp(-bv * D_non)
        return float(np.sum((S['sn'] - pred) ** 2))
    ex, ey = _base(v0)
    c1 = np.array([SOL._fibonacci_cone_point(*v0, *ex, *ey, a1, i, n1) for i in range(n1)])
    s1 = np.array([sse(c) for c in c1]); s0 = sse(v0)
    b1 = c1[np.nanargmin(s1)] if np.nanmin(s1) < s0 else v0
    bx_, by_ = _base(b1)
    c2 = np.array([SOL._fibonacci_cone_point(*b1, *bx_, *by_, a2, i, n2) for i in range(n2)])
    s2 = np.array([sse(c) for c in c2])
    tutti = np.vstack([v0[None], c1, c2]); ss = np.concatenate([[s0], s1, s2])
    scelta = tutti[np.nanargmin(ss)]
    tb, _, _ = SOL.refine_fiber_direction_cone(bv, bc, S['sn'], v0, f_fib, f_res, f_non, 0.0, D_res, D_non, 0.0,
                                                False, a1, n1, a2, n2)
    assert np.allclose(tb, scelta), 'il cono ricalcolato non sceglie la stessa direzione del toolbox'

    def loc(v):
        v = np.atleast_2d(v)
        v = v * np.sign(v @ v0)[:, None]
        return np.degrees(np.arctan2(v @ ex, v @ v0)), np.degrees(np.arctan2(v @ ey, v @ v0))
    fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.5), constrained_layout=True)
    a = axs[0]
    vic = fd[np.argsort(-np.abs(fd @ v0))[:7]]
    a.scatter(*loc(vic), s=70, color='#d9d7d0', label='nodi del dizionario')
    sc = a.scatter(*loc(np.vstack([c1, c2])), c=np.concatenate([s1, s2]), cmap='viridis_r', s=28, zorder=3)
    fig.colorbar(sc, ax=a, label='SSE del candidato')
    a.scatter(*loc(v0), s=110, marker='*', color=INK, label='nodo di partenza', zorder=4)
    a.scatter(*loc(scelta), s=120, facecolors='none', edgecolors=CAT[7], lw=2, label='scelto (= toolbox)', zorder=5)
    if vera is not None:
        a.scatter(*loc(np.asarray(vera)), s=90, marker='x', color=CAT[7], lw=2, label='direzione vera', zorder=5)
    a.set_aspect('equal'); a.set_xlabel('gradi'); a.set_ylabel('gradi')
    a.legend(fontsize=7.5, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3)
    ang = lambda u, v: np.degrees(np.arccos(np.clip(abs(u @ v), 0, 1)))
    a.set_title('Candidati dei due livelli colorati per SSE')
    a = axs[1]
    e = [ang(v0, vera), ang(scelta, vera)] if vera is not None else [np.nan, np.nan]
    a.bar(['nodo di Stage A', 'dopo il cono'], e, color=[INK2, CAT[2]], width=0.55)
    for i, v in enumerate(e):
        a.text(i, v, f'{v:.1f}°', ha='center', va='bottom', fontsize=9)
    a.set_ylabel('errore angolare [gradi]'); a.set_title('Quanto si avvicina alla direzione vera')
    return _salva(fig, file), dict(errore_nodo=e[0], errore_cono=e[1], sse_nodo=s0, sse_scelto=float(np.nanmin(ss)))


# ═════════════════════════════════════════════════════════════════════════════
# 18  crossing: FF ri-stimata e MRDS
# ═════════════════════════════════════════════════════════════════════════════
def ff_contro_kappa(K, ijk, kappas):
    """FF del crossing ri-stimata a ogni kappa per un voxel (None se Stage A non vede due fibre)."""
    bv, bc = K['prot']
    S = stage_a_voxel(K, ijk)
    fd, nb, pairs, iso = (np.asarray(K[k]) for k in ('fiber_dirs', 'neighbor_idx', 'diff_pairs', 'iso_grid'))
    na, nd, npair = K['n_aniso'], int(K['n_dirs']), K['n_pairs']
    idx, _ = SOL.select_dominant_directions(S['w2'][:na], nd, npair, nb, fd, 2, K['min_weight_fraction'],
                                            K['min_separation_cos'], K['min_peak_ratio'],
                                            K['min_dominant_concentration'])
    if idx[1] < 0:
        return None
    f_fib = _frazioni_stage_a(K, S['w2'])[0]
    sv = float(np.asarray(K['sigma_map'])[tuple(ijk)])
    out = []
    for kap in kappas:
        io_ = np.empty(len(bv))
        out.append(SOL.crossing_ff_resolve(S['sn'], bv, bc, fd, fd[idx[:2]], 2, pairs, iso,
                                           kap * (sv / S['s0']) ** 2, io_) if kap > 0 else f_fib)
    return np.array(out)


def fig_crossing(K, ijk, verita=None, file=None, kappas=(0, 1, 3, 10, 30, 100, 300, 1000), altri=None):
    bv, bc = K['prot']
    S = stage_a_voxel(K, ijk)
    fd, nb, pairs, iso = (np.asarray(K[k]) for k in ('fiber_dirs', 'neighbor_idx', 'diff_pairs', 'iso_grid'))
    na, nd, npair = K['n_aniso'], int(K['n_dirs']), K['n_pairs']
    idx, wts = SOL.select_dominant_directions(S['w2'][:na], nd, npair, nb, fd, 2, K['min_weight_fraction'],
                                              K['min_separation_cos'], K['min_peak_ratio'],
                                              K['min_dominant_concentration'])
    assert idx[1] >= 0, 'il voxel scelto non e\' un crossing per Stage A'
    dirs = fd[idx[:2]]
    f_fib, f_res, f_non, D_res, D_non = _frazioni_stage_a(K, S['w2'])
    quote = wts[:2] / wts[:2].sum()
    sigma_v = float(np.asarray(K['sigma_map'])[tuple(ijk)])
    ff_k = []
    for kap in kappas:
        iso_out = np.empty(len(bv))
        ff = SOL.crossing_ff_resolve(S['sn'], bv, bc, fd, dirs, 2, pairs, iso, kap * (sigma_v / S['s0']) ** 2, iso_out) \
            if kap > 0 else f_fib
        ff_k.append(ff)
    iso_out = np.empty(len(bv))
    kap0 = K['crossing_ff_kappa']
    ff_r = SOL.crossing_ff_resolve(S['sn'], bv, bc, fd, dirs, 2, pairs, iso, kap0 * (sigma_v / S['s0']) ** 2, iso_out)
    fr = quote * ff_r
    traiett = []
    for n_lm in range(0, 26):
        AD, RD = SOL.estimate_AD_RD_mrds(bv, bc, S['sn'], dirs, fr, iso_out, 3, n_lm)
        E = np.exp(-bv[:, None] * (RD[None] + (AD - RD)[None] * (bc @ dirs.T) ** 2))
        C = float(np.sum((S['sn'] - iso_out - E @ fr) ** 2))
        traiett.append(dict(it=n_lm, C=C, AD1=AD[0], RD1=RD[0], AD2=AD[1], RD2=RD[1]))
    T = pd.DataFrame(traiett)
    fin = K['res'][tuple(ijk)]
    fig, axs = plt.subplots(1, 3, figsize=(14.5, 4.1), constrained_layout=True)
    a = axs[0]
    a.semilogx([max(k, 0.3) for k in kappas], ff_k, 'o-', color=INK, lw=1.8, label='FF ri-stimata (questo voxel)')
    a.axhline(f_fib, color=INK2, ls=':', label=f'FF di Stage A {f_fib:.2f}')
    if verita is not None:
        a.axhline(verita['FF'], color=CAT[2], ls='--', label=f"FF vera {verita['FF']:.2f}")
    if altri:
        F = np.array([f for f in (ff_contro_kappa(K, v, kappas) for v in altri) if f is not None])
        if len(F):
            xk = [max(k, 0.3) for k in kappas]
            a.fill_between(xk, np.percentile(F, 10, 0), np.percentile(F, 90, 0), color=CAT[0], alpha=0.18, lw=0,
                           label=f'tutti i crossing ({len(F)} voxel): 10–90°')
            a.plot(xk, np.median(F, 0), color=CAT[0], lw=1.6, label='tutti i crossing: mediana')
    a.axvline(kap0, color=CAT[1], lw=1); a.text(kap0 * 1.1, min(ff_k), f'κ = {kap0:g}', color=CAT[1], fontsize=8)
    a.set_xlabel('κ (κ=0 disegnato a 0.3: tiene la FF di Stage A)'); a.set_ylabel('FF del crossing')
    a.set_title('La FF del crossing sul supporto rilevato'); a.legend(fontsize=7.5)
    a = axs[1]
    a.semilogy(T.it, T.C, 'o-', color=INK, lw=1.6, ms=4)
    a.set_xlabel('iterazioni di Levenberg–Marquardt (dopo 3 alternate)'); a.set_ylabel('costo C(p)')
    a.set_title('MRDS: il costo scende e si ferma')
    a = axs[2]
    for k, c, ls in (('AD1', CAT[0], '-'), ('RD1', CAT[0], '--'), ('AD2', CAT[1], '-'), ('RD2', CAT[1], '--')):
        a.plot(T.it, T[k] * 1e3, ls, color=c, lw=1.8, label=k.replace('1', ' pop1').replace('2', ' pop2'))
    if verita is not None:
        for k, c in (('AD', INK2), ('RD', INK2)):
            for v in verita[k]:
                a.axhline(v * 1e3, color=c, lw=0.8, ls=':')
    a.set_xlabel('iterazioni LM'); a.set_ylabel('[e-3 mm²/s]'); a.set_title('I quattro parametri (punteggiato = veri)')
    a.legend(fontsize=7.5, ncol=2)
    return _salva(fig, file), T, dict(FF_stageA=f_fib, FF_ristimata=ff_r, kappa=kap0,
                                      ridge=kap0 * (sigma_v / S['s0']) ** 2, quote=quote.tolist(),
                                      AD_finale=[fin[IX['axial_diffusivity_pop1']], fin[IX['axial_diffusivity_pop2']]],
                                      RD_finale=[fin[IX['radial_diffusivity_pop1']], fin[IX['radial_diffusivity_pop2']]])


# ═════════════════════════════════════════════════════════════════════════════
# 18b  il cono per popolazione nei crossing (opzione crossing_dir_refine)
# ═════════════════════════════════════════════════════════════════════════════
def cono_crossing_voxel(K, ijk):
    """Ripete per un voxel di crossing il percorso del kernel: FF ri-stimata sul supporto, MRDS sulle direzioni
    della griglia, poi il cono per popolazione (refine_crossing_directions_cone). Ritorna prima e dopo."""
    bv, bc = K['prot']
    S = stage_a_voxel(K, ijk)
    fd, nb, pairs, iso = (np.asarray(K[k]) for k in ('fiber_dirs', 'neighbor_idx', 'diff_pairs', 'iso_grid'))
    na, nd, npair = K['n_aniso'], int(K['n_dirs']), K['n_pairs']
    idx, wts = SOL.select_dominant_directions(S['w2'][:na], nd, npair, nb, fd, 2, K['min_weight_fraction'],
                                              K['min_separation_cos'], K['min_peak_ratio'],
                                              K['min_dominant_concentration'])
    assert idx[1] >= 0, "il voxel scelto non e' un crossing per Stage A"
    dirs = fd[idx[:2]].copy()
    sigma_v = float(np.asarray(K['sigma_map'])[tuple(ijk)])
    iso_out = np.empty(len(bv))
    ff_r = SOL.crossing_ff_resolve(S['sn'], bv, bc, fd, dirs, 2, pairs, iso,
                                   K['crossing_ff_kappa'] * (sigma_v / S['s0']) ** 2, iso_out)
    fr = wts[:2] / wts[:2].sum() * ff_r
    AD, RD = SOL.estimate_AD_RD_mrds(bv, bc, S['sn'], dirs, fr, iso_out, M._MRDS_INIT_N_ITER, M._MRDS_LM_MAX_ITER)
    d2, AD2, RD2 = SOL.refine_crossing_directions_cone(
        bv, bc, S['sn'], dirs, fr, iso_out, AD, RD, K['min_separation_cos'], K['cone1_half_angle'],
        int(K['n1_cone']), K['cone2_half_angle'], int(K['n2_cone']), M._MRDS_CONE_LM_ITER)
    return dict(dirs0=dirs, AD0=AD, RD0=RD, dirs1=d2, AD1=AD2, RD1=RD2,
                sse0=float(SOL._mrds_sse(bv, bc, S['sn'], dirs, fr, iso_out, AD, RD)),
                sse1=float(SOL._mrds_sse(bv, bc, S['sn'], d2, fr, iso_out, AD2, RD2)))


def crossing_casuali(prot, n, snr, lam, seed=0, opzioni=None):
    """n crossing bilanciati (0.30 + 0.30, AD 1.7 RD 0.4 e-3, 60-90 gradi) con orientamento CASUALE: errore
    angolare delle due popolazioni dopo il fit a lambda congelati (NaN dove il fit non vede il crossing)."""
    rng = np.random.default_rng(seed)
    bv, bc = prot
    A = np.zeros((n, 1, 1, len(bv)), np.float32)
    vere = np.full((n, 1, 1, 2, 3), np.nan)
    for i in range(n):
        u = rng.normal(size=3); u /= np.linalg.norm(u)
        v = ruota_verso(u, rng.uniform(60, 90), rng.normal(size=3))
        S = segnale(prot, [(0.3, u, 1.7e-3, 0.4e-3), (0.3, v, 1.7e-3, 0.4e-3)], dict(R=0.1, H=0.2, W=0.1), D_NOM)
        A[i, 0, 0] = S / S[bv < 100].mean()
        vere[i, 0, 0] = u, v
    Mm, sg = rumore_rician(A, snr, seed + 1)
    m = DBSI_Adaptive(lambda_aniso=lam[0], lambda_iso=lam[1], n_iso=6, **(opzioni or {}))
    with contextlib.redirect_stdout(io.StringIO()):
        res, _ = m.fit(Mm, bv, bc, np.ones(A.shape[:3], bool), run_calibration=False,
                       noise_map=np.full(A.shape[:3], sg))
    return errore_angolare(res, vere).reshape(n, 2)


def _ang(u, v):
    return float(np.degrees(np.arccos(np.clip(abs(np.dot(u, v)), 0, 1))))


def fig_cono_crossing(K, C, vere, tab_err, file=None):
    """C = cono_crossing_voxel; vere = (2, 3) direzioni vere; tab_err: scenario, popolazione, braccio, mediana."""
    fd = np.asarray(K['fiber_dirs'])
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True,
                            gridspec_kw=dict(width_ratios=[1, 1, 1.5]))
    for k in (0, 1):
        a = axs[k]
        t = vere[np.argmax([abs(C['dirs0'][k] @ v) for v in vere])]      # la vera piu' vicina a questa popolazione
        ex, ey = _base(t)

        def loc(v):
            v = np.atleast_2d(v); v = v * np.sign(v @ t)[:, None]
            return np.degrees(np.arctan2(v @ ex, v @ t)), np.degrees(np.arctan2(v @ ey, v @ t))
        vic = fd[np.argsort(-np.abs(fd @ t))[:7]]
        a.scatter(*loc(vic), s=70, color='#d9d7d0', label='nodi del dizionario')
        a.scatter(*loc(C['dirs0'][k]), s=130, marker='*', color=INK, zorder=4,
                  label=f"Stage A + MRDS ({_ang(C['dirs0'][k], t):.1f}°)")
        a.scatter(*loc(C['dirs1'][k]), s=120, facecolors='none', edgecolors=CAT[2], lw=2.2, zorder=5,
                  label=f"dopo il cono ({_ang(C['dirs1'][k], t):.1f}°)")
        a.scatter(0, 0, s=90, marker='x', color=CAT[7], lw=2, zorder=5, label='direzione vera')
        a.set_aspect('equal'); a.set_xlabel('gradi'); a.set_ylabel('gradi')
        a.set_title(f'popolazione {k + 1}: AD/RD {C["AD0"][k]*1e3:.2f}/{C["RD0"][k]*1e3:.2f} → '
                    f'{C["AD1"][k]*1e3:.2f}/{C["RD1"][k]*1e3:.2f}', fontsize=9)
        a.legend(fontsize=7.5, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
    a = axs[2]
    sc = list(dict.fromkeys(tab_err.scenario))
    x = np.arange(len(sc) * 2)
    for j, (br, col) in enumerate((('griglia', INK2), ('cono', CAT[2]))):
        v = [tab_err[(tab_err.scenario == s) & (tab_err.popolazione == q) & (tab_err.braccio == br)].mediana.iloc[0]
             for s in sc for q in (1, 2)]
        a.bar(x + (j - 0.5) * 0.38, v, width=0.38, color=col, label=br)
    a.set_xticks(x, [f'{s}\npop {q}' for s in sc for q in (1, 2)], fontsize=7.5)
    a.set_ylabel('errore angolare mediano [gradi]'); a.legend(fontsize=8)
    a.set_title('Errore mediano: crossing della batteria e orientamenti casuali')
    fig.suptitle(f"Il cono per popolazione: SSE {C['sse0']:.2e} → {C['sse1']:.2e} su questo voxel", fontsize=10)
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 18c  il test 2-contro-1 (opzione two_vs_one_threshold)
# ═════════════════════════════════════════════════════════════════════════════
def fig_test21(stat, npop, T, snr_fig, file=None):
    """stat: DataFrame SNR, vero (popolazioni vere), stat21 dei candidati. npop: SNR, vero, fit ('1.8' o
    'test'), due_pct (quota dei voxel con 2 popolazioni stimate)."""
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.1), constrained_layout=True)
    a = axs[0]
    tr = lambda x: np.sign(x) * np.log10(1 + np.abs(x))          # logaritmo con segno: stat21 puo' essere < 0
    d = stat[stat.SNR == snr_fig]
    b = np.linspace(tr(min(-10, d.stat21.min())), tr(max(1e3, d.stat21.max())), 60)
    for v, col, nome in ((1, CAT[0], 'una fibra vera'), (2, CAT[1], 'due fibre vere')):
        q = d[d.vero == v].stat21.to_numpy()
        if q.size:
            a.hist(tr(q), bins=b, color=col, alpha=0.55, weights=np.full(q.size, 100 / q.size),
                   label=f'{nome} (n={q.size})')
    a.axvline(tr(T), color=INK, ls='--', lw=1.2)
    a.text(tr(T) * 1.05, a.get_ylim()[1] * 0.9, f'T = {T:g}', fontsize=8)
    tk = [v for v in (-10, -1, 0, 1, 10, 100, 1000, 10000) if b[0] <= tr(v) <= b[-1]]
    a.set_xticks([tr(v) for v in tk], [str(v) for v in tk])
    a.set_xlabel('stat21 = (RSS_1pop − RSS_2pop) / σ²  (scala log con segno)'); a.set_ylabel('% dei voxel')
    a.set_title(f'I candidati crossing, SNR {snr_fig}'); a.legend(fontsize=7.5)
    for a, v, tit in ((axs[1], 1, 'Falsi crossing (una fibra vera)'), (axs[2], 2, 'Crossing trovati (due fibre vere)')):
        for fit_, ls in (('1.8', '--'), ('test', '-')):
            s = npop[(npop.vero == v) & (npop.fit == fit_)].sort_values('SNR')
            a.plot(s.SNR, s.due_pct, ls, marker='o', color=INK if fit_ == '1.8' else CAT[2], lw=1.8,
                   label='min_peak_ratio 0.35 (fino alla 1.8)' if fit_ == '1.8' else f'test 2-contro-1, T = {T:g} (1.9.0)')
        a.set_xlabel('SNR'); a.set_ylabel('% con 2 popolazioni stimate'); a.set_title(tit); a.set_ylim(0, 100)
        a.grid(alpha=0.5)
    axs[1].legend(fontsize=7.5)
    return _salva(fig, file)


# ═════════════════════════════════════════════════════════════════════════════
# 23  incertezza, spiegata semplice
# ═════════════════════════════════════════════════════════════════════════════
def fig_bersaglio(stime, se, vero, nome_x, nome_y, titoli, file=None):
    """stime: lista di array (n, 2) per gruppo; se: lista di (se_x, se_y) mediane; vero: lista di (x, y)."""
    fig, axs = plt.subplots(1, len(stime), figsize=(5.2 * len(stime), 4.6), constrained_layout=True)
    axs = np.atleast_1d(axs)
    t = np.linspace(0, 2 * np.pi, 200)
    for a, s, e, v, tit in zip(axs, stime, se, vero, titoli):
        s = s[np.all(np.isfinite(s), 1)]
        a.scatter(s[:, 0], s[:, 1], s=10, color=CAT[0], alpha=0.5, label='stime (una per realizzazione del rumore)')
        m = s.mean(0)
        a.plot(v[0] + 1.96 * e[0] * np.cos(t), v[1] + 1.96 * e[1] * np.sin(t), color=CAT[2], lw=2,
               label='±1.96 SE del toolbox attorno al vero')
        a.plot(*v, marker='*', ms=16, color=INK, ls='', label='verità')
        a.plot(*m, marker='o', ms=9, color=CAT[7], ls='', label='media delle stime')
        a.annotate('', xy=m, xytext=v, arrowprops=dict(arrowstyle='->', color=CAT[7], lw=1.5))
        sd = s.std(0)
        a.set_title(f'{tit}\nbias ({m[0]-v[0]:+.3f}, {m[1]-v[1]:+.3f})   sd ({sd[0]:.3f}, {sd[1]:.3f})   SE ({e[0]:.3f}, {e[1]:.3f})',
                    fontsize=8.5)
        a.set_xlabel(nome_x); a.set_ylabel(nome_y)
    axs[0].legend(fontsize=7.5, loc='upper left')
    return _salva(fig, file)


def fig_se_protocolli(tab, file=None):
    """tab: colonne protocollo, SNR, metrica, se (mediana)."""
    met = list(dict.fromkeys(tab.metrica))
    fig, axs = plt.subplots(1, len(met), figsize=(4.0 * len(met), 3.5), constrained_layout=True)
    axs = np.atleast_1d(axs)
    for a, k in zip(axs, met):
        for i, p in enumerate(dict.fromkeys(tab.protocollo)):
            s = tab[(tab.metrica == k) & (tab.protocollo == p)].sort_values('SNR')
            a.plot(s.SNR, s.se, 'o-', color=CAT[i], lw=1.8, label=p)
        a.set_xlabel('SNR'); a.set_ylabel('SE mediana'); a.set_title(k)
        a.grid(alpha=0.5)
    axs[-1].legend(fontsize=8)
    fig.suptitle('La SE dipende da protocollo e SNR, non dal tessuto stimato bene o male', fontsize=10)
    return _salva(fig, file)


def fig_residuo(rr_ok, rr_male, soglia, file=None):
    fig, a = plt.subplots(figsize=(7.5, 3.6), constrained_layout=True)
    b = np.linspace(0.4, max(3, np.nanpercentile(rr_male, 99)), 60)
    a.hist(rr_ok[np.isfinite(rr_ok)], bins=b, color=CAT[0], alpha=0.6, density=True, label='il modello descrive il voxel')
    a.hist(rr_male[np.isfinite(rr_male)], bins=b, color=CAT[7], alpha=0.6, density=True, label='il modello NON lo descrive')
    a.axvline(1, color=INK, lw=1); a.axvline(soglia, color=CAT[7], ls='--')
    a.text(soglia * 1.02, a.get_ylim()[1] * 0.85, 'flag 16', color=CAT[7], fontsize=8)
    a.set_xlabel('residuo / σ  (≈ 1 se modello e σ sono giusti)'); a.set_ylabel('densità')
    a.set_title('residual_over_sigma: il controllo che il modello basti')
    a.legend(fontsize=8)
    return _salva(fig, file)


def soglia_residuo(dof):
    """La soglia di residual_over_sigma del flag 16 (quantile 99.9% di chi2_dof / dof, radice)."""
    from dbsi_toolbox import uncertainty as U
    h = 2.0 / (9.0 * dof)
    return float(np.sqrt((1.0 - h + U._RESID_FLAG_Z * np.sqrt(h)) ** 3))


def fit_voxel_ripetuti(prot, fibre, iso, n, snr, lam, seed=0, D=None):
    """n voxel con la STESSA verita' e rumore Rician indipendente; fit di produzione a lambda congelati.
    Ritorna (risultati (n, canali), se (n, canali), flags (n,), residuo/sigma (n,))."""
    D = D or dict(D_NOM)
    S = segnale(prot, fibre, iso, D)
    S = S / (sum(f for f, *_ in fibre) + sum(iso.values()))
    A = np.repeat(S[None, None, None, :], n, axis=0).astype(np.float32)
    Mm, sg = rumore_rician(A, snr, seed)
    m = DBSI_Adaptive(lambda_aniso=lam[0], lambda_iso=lam[1], n_iso=6)
    with contextlib.redirect_stdout(io.StringIO()):
        res, _ = m.fit(Mm, prot[0], prot[1], np.ones(A.shape[:3], bool), run_calibration=False,
                       noise_map=np.full(A.shape[:3], sg))
    u = m.uncertainty_
    return (res.reshape(n, -1), u['se'].reshape(n, -1), u['flags'].reshape(n),
            u['residual_over_sigma'].reshape(n))

In [ ]:
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore', category=RuntimeWarning)
stile()
VER = dbsi_toolbox.__version__
assert tuple(int(x) for x in VER.split('.')[:3]) >= (1, 9, 0), f'serve pyDBSI >= 1.9.0, trovata {VER}'
OPZ_18 = dict(crossing_dir_refine=False, two_vs_one_threshold=None)   # i crossing come fino alla 1.8 (§14-18)
FIG = Path.cwd() / 'risultati_matematica' / f'v{VER.replace(".", "")}'
FIG.mkdir(parents=True, exist_ok=True)

P3_BVAL = """0 0 0 0 0 0 0 0 0 300 300 300 700 700 700 700 700 700 700 700 700 700 700 700 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000"""
P3_BVEC = """0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 -0.827892 -0.443607 0.388907 -0.195963 -0.021996 -0.772169 0.428901 -0.482508 -0.400001 0.617707 -0.971440 0.439716 0.074098 -0.770731 0.947588 0.608595 0.842131 0.215671 0.820813 0.526317 0.116272 0.998930 -0.041864 0.466815 0.706768 -0.330802 0.873779 0.029363 -0.077901 0.845481 -0.543135 0.465775 0.265880 -0.577315 0.758291 0.436608 0.283505 0.704573 0.140756 0.476272 0.609103 -0.048362 0.806042 0.180569 0.162519 0.848290 -0.221219 0.641808 0.439881 -0.207121 0.956201 -0.155265 0.268266 0.760663 -0.455737 0.828128 0.129733 -0.184321 0.974578 -0.468655 0.464696 0.519300 -0.550940 0.956542 -0.229050 -0.033559 0.850511 -0.695282 0.671407 0.177585 -0.506768 0.976012 -0.562832 0.187547 0.596676 -0.797793 0.821416 -0.202643 -0.348672 0.863482 -0.812656 0.414584 0.250297
0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 -0.533708 0.295357 -0.913022 -0.734983 -0.999750 -0.459459 -0.661673 -0.764722 -0.068022 -0.777409 -0.189738 -0.054967 -0.625809 0.349569 -0.263092 -0.166643 -0.426494 -0.509055 0.201112 -0.825043 0.029624 -0.046251 -0.844528 0.547826 -0.657461 -0.255804 0.485365 -0.999562 0.525752 -0.158235 -0.648146 0.848154 -0.817423 0.198014 0.438381 -0.787298 -0.624617 -0.578781 -0.895757 -0.353697 -0.785226 -0.690718 -0.279942 -0.978898 -0.296732 -0.525955 -0.865470 -0.043836 -0.880729 -0.431172 -0.172858 -0.986372 0.023712 -0.595628 -0.665170 0.135875 -0.942836 -0.102248 -0.195813 -0.870073 0.285828 -0.696926 -0.362969 0.195848 -0.926285 0.226112 -0.294319 -0.641060 0.454250 -0.769284 -0.011808 0.154341 -0.806667 0.500933 -0.414886 -0.326046 0.516070 -0.767736 0.334839 0.046067 -0.582729 0.686266 -0.505591
0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 -0.172482 0.846154 -0.123058 0.649153 -0.003912 0.438922 0.615006 -0.427066 0.913987 -0.118627 -0.142488 0.896453 -0.776448 0.532705 0.181270 0.775785 0.330026 0.833276 0.534621 0.205656 0.992776 -0.000113 0.533872 0.694241 -0.261198 0.908369 0.030521 0.003816 0.847063 -0.510024 0.533771 0.252367 -0.511006 0.792147 -0.482511 0.435357 0.727653 0.410596 0.421672 0.805024 0.111416 0.721505 0.521467 0.095675 0.941030 0.061435 0.449471 0.765611 -0.175560 0.878175 0.236219 0.054427 0.963053 -0.258108 0.591484 0.543822 -0.306969 0.977533 -0.108878 0.152761 0.838069 -0.494592 0.751478 0.216037 -0.299219 0.973523 -0.435899 0.325000 0.585550 -0.613731 0.862001 -0.153558 -0.180301 0.844922 -0.686912 0.507169 0.242793 -0.607879 0.875392 -0.502271 0.004084 0.597628 -0.825669"""
def _da_testo(a, b):
    import io as _io
    bv = np.loadtxt(_io.StringIO(a)).ravel(); bc = np.loadtxt(_io.StringIO(b)); bc = bc.T if bc.shape[0] == 3 else bc
    n = np.linalg.norm(bc, axis=1, keepdims=True); n[n == 0] = 1
    return bv, bc / n
P3 = _da_testo(P3_BVAL, P3_BVEC)
print(f'pyDBSI {VER}; P3: {descrivi_protocollo(P3)}; figure in {FIG}')

# fantoccio della batteria (13 scenari) per le sezioni che hanno bisogno della verita'
A_b, TRU, LAB, NOMI_SC = fantoccio_batteria(P3, blocco=(6, 6, 4), colonne=7, d_variabile=False, seed=0)
MB, SIG_B = rumore_rician(A_b, SNR_FANTOCCIO, seed=1)
MASK_B = LAB >= 0
def voxel_di(nome, q=0):
    s = NOMI_SC.index(nome); c = np.argwhere((LAB == s) & interno(LAB)); return tuple(c[q])
def dir_vere(ijk):
    v = TRU['dirs'][ijk]; return v[np.isfinite(v[:, 0])] if np.isfinite(v[0, 0]) else None

## §3.3 — Protocollo: 3-ISO o 2-ISO

**L'idea.** Ogni compartimento isotropo decade con la sua diffusività: `S(b) = exp(−b·D)`. A b = 2000
l'acqua libera (D = 3.0) è già sotto il rumore (e⁻⁶ ≈ 0.0025), mentre la hindered (D ≈ 1.0) vale ancora 0.14.
È solo **a b alto** che le due curve si separano; a b basso si somigliano. Per questo il toolbox chiede
b_max ≥ 2000 e almeno due gusci prima di stimare acqua e hindered separatamente (3-ISO); altrimenti le
fonde in una "non-restricted" (2-ISO).

**Cosa guardare.** A sinistra, dove cadono i gusci del P3 rispetto alle tre curve e al livello del rumore;
a destra, la differenza hindered − acqua: è sopra 2σ in tutta la zona dei b alti del P3.

In [ ]:
fig_protocollo(P3, SNR_FANTOCCIO, protocolli={'P3': P3}, file=FIG / '01_protocollo_3iso.png'); plt.show()
print(M.analyse_protocol(P3[0])[3])

## §4 — Il rumore e la correzione Rician

**L'idea.** La magnitudine è il modulo di un segnale complesso con rumore sulle due componenti. Quando il
segnale vero A è grande, il rumore si somma e si sottrae in modo simmetrico. Quando A è piccolo (alto b), il
modulo **non può essere negativo**: la misura resta "sollevata" su un pavimento di circa 1.25σ anche se il
segnale vero è zero. Il secondo momento dice di quanto: E[M²] = A² + 2σ². Il toolbox toglie 2σ² a ogni misura
(con la σ del voxel, dalla mappa di rumore) e tronca a zero.

**Cosa guardare.** Al centro, la curva rossa (grezzo) che non scende sotto il pavimento e la verde
(corretta) che torna sulla diagonale. A destra, il liquor: senza correzione il pavimento ad alto b sembra un
plateau, cioè acqua ristretta; la tabellina dice quanta RF ne inventerebbe Stage D.

In [ ]:
fig_rician(SNR_FANTOCCIO, P3, file=FIG / '02_rician.png'); plt.show()

## §5.3 — Lo spettro isotropo: la griglia ancorata, e la matrice A

**L'idea.** Il blocco isotropo è una serie di esponenziali, una per diffusività della griglia. I punti sono
log-uniformi (l'esponenziale è ugualmente sensibile a ogni decade di D), poi se ne aggiunge uno appena dentro
ciascuna soglia di classe (0.3 e 3.0), così nessuna classe resta senza colonna al bordo, e si riempiono i buchi
troppo larghi (rapporto > 2) con la media geometrica. Da n_iso = 6 escono 11 colonne.

**La matrice A del P3.** Ogni colonna è un "atomo": il segnale che darebbe un compartimento puro su tutte le 92
misure. Le colonne anisotrope sono una per coppia (AD, RD) e per direzione; le isotrope una per D. Il fit
cerca la combinazione non negativa di colonne che riproduce il segnale del voxel.

**Cosa guardare.** Nella matrice, le righe a b = 0 tutte a 1; le colonne di fibra "a strisce" (il segnale
dipende dall'angolo fra gradiente e fibra), quelle isotrope lisce (dipendono solo da b).

In [ ]:
fig, GRIGLIA = fig_griglia(file=FIG / '03_griglia_isotropa.png'); plt.show()
print('griglia (e-3 mm^2/s):', np.round(GRIGLIA * 1e3, 3))
K = intercetta_fit(P3, MB, MASK_B, np.full(MASK_B.shape, SIG_B), lam=(9.8, 0.015), opzioni=OPZ_18)
print(f"dizionario: {int(K['n_dirs'])} direzioni x {K['n_pairs']} coppie = {K['n_aniso']} colonne di fibra + "
      f"{len(K['iso_grid'])} isotrope; coppie (AD, RD) e-3: {np.round(np.asarray(K['diff_pairs']) * 1e3, 2).tolist()}")
fig_matrice_A(K, file=FIG / '04_matrice_A_P3.png'); plt.show()

## §5.5 — Il grafo dei vicini e lo schema del cono

**Il grafo.** Per trovare i picchi del peso anisotropo serve sapere chi è vicino a chi. Una fibra non ha verso:
v e −v sono la stessa fibra. Vicino all'equatore dell'emisfero, il vicino "vero" di una direzione può stare
dalla parte opposta del cerchio (è il suo −v riportato su). Il toolbox usa |v·v′|: al centro della figura, i
vicini verdi (toolbox) includono quelli oltre l'equatore, i cerchi rossi (prodotto scalare semplice) no.

**Il cono.** Le direzioni del dizionario distano ~16.6°: la fibra vera cade quasi sempre in mezzo. Il cono
prova candidati su una calotta grande quanto la spaziatura, poi su una più piccola attorno al migliore, fino a
una risoluzione di ~1°. Il numero di candidati cresce con l'area della calotta (il quadrato del rapporto degli
angoli).

In [ ]:
fig, INFO_CONO = fig_grafo_e_cono(K, file=FIG / '05_grafo_e_cono.png'); plt.show()
print({k: round(float(v), 2) for k, v in INFO_CONO.items()})

## §6.1 — n_iso non si calibra: la convergenza

**L'idea.** n_iso è la densità della griglia isotropa, non il numero di compartimenti: le frazioni riportate
vengono da Stage D, che usa tre centroidi fissi. Il notebook 06 (un soggetto vero, 3000 voxel, toolbox 1.3.6) ha
rifatto il fit a dieci densità. **Risultato:** le frazioni non si muovono; il tensore sta entro la banda di
tolleranza da n_iso = 6; oltre, cresce solo il costo (e le NNLS che non convergono). La tabella è quella del 06.
Sotto, la stessa domanda sul fantoccio col toolbox attuale.

In [ ]:
display(CONVERGENZA_06)
righe = []
A_s, TRU_s, LAB_s, _ = fantoccio_batteria(P3, blocco=(4, 4, 2), colonne=7, d_variabile=False, seed=3)
M_s, sg_s = rumore_rician(A_s, SNR_FANTOCCIO, seed=4)
for n_iso in (4, 6, 8, 12):
    m = DBSI_Adaptive(lambda_aniso=9.8, lambda_iso=0.015, n_iso=n_iso)
    with contextlib.redirect_stdout(io.StringIO()):
        res, _ = m.fit(M_s, P3[0], P3[1], LAB_s >= 0, run_calibration=False, noise_map=np.full(LAB_s.shape, sg_s))
    sel = (LAB_s >= 0) & interno(LAB_s)
    r = dict(n_iso=n_iso, colonne=len(m.dictionary_['iso_grid']))
    for k, c in (('FF', 'fiber_fraction'), ('RF', 'restricted_fraction'), ('HF', 'hindered_fraction'),
                 ('WF', 'water_fraction'), ('RDw', 'radial_diffusivity_weighted')):
        e = res[..., IX[c]][sel] - TRU_s[k][sel]
        r[k] = float(np.nanmean(np.abs(e))) * (1e3 if k == 'RDw' else 1)
    righe.append(r)
SINT = pd.DataFrame(righe); display(SINT.round(4))
fig_convergenza(SINT, file=FIG / '06_convergenza_niso.png'); plt.show()

## §7 — Il campione di calibrazione e il suo σ

**L'idea.** La calibrazione non usa tutto il cervello: estrae 1000 voxel (seme fisso, quindi riproducibile) e
ne calcola il rumore normalizzato σ/S0. Quel rumore è molto diverso da voxel a voxel (dipende da S0: un voxel
scuro è più rumoroso). I criteri della calibrazione confrontano **la media dei residui al quadrato** con Nσ²:
il σ giusto per quel confronto è la **media quadratica** (RMS) dei σ/S0, non la mediana. Con la mediana il
bersaglio era troppo piccolo (sui dati veri RMS² ≈ 3.5 × mediana²).

**Due domande.** (a) RMS e mediana sono stabili da un campione all'altro? (b) λ cambia se cambia il campione,
e con quanti voxel smette di cambiare? Per la (b) si rifà la calibrazione vera con 5 semi diversi a 4 dimensioni.

In [ ]:
def _carica_reale():
    import nibabel as nib
    p = Path(REALE['prep']); s = REALE['stem']
    f = dict(dwi=p / f'{s}_preprocessed_N4.nii.gz', bval=p / f'{s}_corrected.bval', bvec=p / f'{s}_corrected.bvec',
             mask=p / f'{s}_preprocessed_brain_mask.nii.gz', noise=p / f'{s}_noise_N4.nii.gz')
    if not all(v.is_file() for v in f.values()):
        return None
    from dbsi_toolbox import load_data
    data, aff, bv, bc, mask = load_data(str(f['dwi']), str(f['bval']), str(f['bvec']), mask_path=str(f['mask']), verbose=False)
    nm = np.asarray(nib.load(str(f['noise'])).dataobj, dtype=np.float64)
    return (np.asarray(bv, float), np.asarray(bc, float)), np.asarray(data, np.float32), np.asarray(mask, bool), np.squeeze(nm)

REAL = _carica_reale()
if REAL is not None:
    PROT_C, DATA_C, MASK_C, NOISE_C = REAL
    FONTE = f"acquisizione vera ({REALE['prep']})"
else:
    PROT_C = P3
    A_c, MASK_C, _ = volume_composito(P3, piccolo=True)
    DATA_C, s_c = rumore_rician(A_c, SNR_FANTOCCIO, seed=7)
    NOISE_C = np.full(MASK_C.shape, s_c)
    FONTE = 'fantoccio composito (nessuna acquisizione vera trovata)'
print('dati della calibrazione:', FONTE)
dc = np.sqrt(np.maximum(DATA_C.astype(np.float64) ** 2 - 2 * NOISE_C[..., None] ** 2, 0))   # come il toolbox
r = rapporti_sigma(dc, MASK_C, PROT_C[0], NOISE_C)
fig, D_SIG = fig_sigma_campione(r, file=FIG / '07_sigma_campione.png'); plt.show()
print(f'mediana {np.median(r):.4f}, RMS {np.sqrt(np.mean(r**2)):.4f}, RMS^2/mediana^2 = {np.mean(r**2)/np.median(r)**2:.2f}')

In [ ]:
if STABILITA_LAMBDA:
    righe = []
    for n in N_LIST_LAMBDA:
        for s in SEMI_LAMBDA:
            t0 = time.time()
            Rs = registra_calibrazione(PROT_C, DATA_C, MASK_C, NOISE_C, n_cal=n, seed=s)
            righe.append(dict(n=n, seme=s, lambda_aniso=Rs['lambda_aniso'], lambda_iso=Rs['lambda_iso'],
                              sigma_cal=Rs['campione']['sigma_cal'], secondi=round(time.time() - t0)))
            print(f"  {n:>5} voxel, seme {s}: lambda_aniso {Rs['lambda_aniso']:.4g}  lambda_iso {Rs['lambda_iso']:.4g}  "
                  f"({righe[-1]['secondi']} s)", flush=True)
    STAB = pd.DataFrame(righe); STAB.to_csv(FIG / '07_lambda_vs_campione.csv', index=False)
    fig_lambda_vs_campione(STAB, file=FIG / '07_lambda_vs_campione.png'); plt.show()

## §8–9 — La calibrazione passo per passo

**Il percorso**, su un'acquisizione (con seme 0, come in produzione):

1. **GCV del blocco isotropo sul segnale pieno** → un λ_iso preliminare. La GCV stima l'errore di
   predizione senza rifare il fit: cerca il λ che bilancia "spiegare i dati" e "non inseguire il rumore".
2. **λ_aniso preliminare per discrepanza**: il residuo medio R(λ) cresce con λ; si cerca per bisezione il λ
   a cui il residuo è grande quanto il rumore (Nσ²). I punti arancioni sono le valutazioni della bisezione.
3. **Il residuo isotropo**: con i due λ preliminari si toglie a ogni voxel la parte di fibra. Resta ciò che il
   blocco isotropo deve spiegare (nel voxel con fibra, la linea tratteggiata è molto più bassa del segnale).
4. **GCV sul residuo isotropo** → λ_iso^GCV.
5. **Tetto di discrepanza**: il λ_iso più grande per cui il blocco isotropo spiega ancora il residuo fino al
   rumore (curva verde contro la linea punteggiata). λ_iso = min(GCV, tetto). Ad alto SNR il tetto è inerte.
9. **λ_aniso finale con la GCV ridge**: la curva GCV e, a destra, i gradi di libertà efficaci: quanti
   parametri "veri" usa il fit a quel λ (da quasi N a λ piccolo a pochi a λ grande).

Il λ_aniso preliminare (passo 2) e quello finale (passo 9) **non devono coincidere**: il primo serve solo a
togliere la fibra al passo 3. La tabella sotto la figura è l'"esempio numerico" del report, rifatto qui.

In [ ]:
R0 = registra_calibrazione(PROT_C, DATA_C, MASK_C, NOISE_C, n_cal=1000)
figs, RIEP = fig_calibrazione(R0, PROT_C, file_prefix=str(FIG / '08_calibrazione'))
plt.show()
print('dati:', FONTE)
display(pd.Series(RIEP).to_frame('valore'))

## §14 — Concentrazione angolare e modulazione di λ_aniso

**L'idea.** Dopo la prima soluzione di Stage A si guarda **come** è distribuito il peso anisotropo sulle
direzioni. Una fibra vera concentra il peso attorno alla sua direzione (un bacino grande: c alto). Un voxel
senza fibre "perde" un po' di segnale isotropo nel blocco anisotropo, sparpagliato su tante direzioni (tanti
bacini piccoli: c basso). Se c è basso, il voxel viene risolto di nuovo con λ_aniso moltiplicato per g(c): la
massa sparpagliata torna nello spettro isotropo.

**Cosa guardare.** Una colonna per tipo di voxel: in alto la prima soluzione, in basso dopo la modulazione
(area dei punti = peso della direzione, colore = bacino, × = fibra vera). Nella WM sana c = 1 e non cambia
niente; nella GM e nel "solo restricted" il peso è sparpagliato, la modulazione abbassa la FF. Sotto, la rampa
g(c) con le distribuzioni di c nei voxel con e senza fibra vera. I valori di c coincidono con il canale
`dominant_basin_concentration` del toolbox (controllato nella cella).

In [ ]:
SCEL = ['WM sana', 'WM demielinizzata', 'Crossing bil.', 'GM', 'Solo restricted']
V = [(n, voxel_di(n, 3), dir_vere(voxel_di(n, 3))) for n in SCEL]
fig, T14 = fig_concentrazione(K, V, file=FIG / '09_concentrazione.png'); plt.show()
c_tb = [float(K['res'][v[1]][IX['dominant_basin_concentration']]) for v in V]
assert np.allclose(T14.c_dopo, c_tb, atol=1e-6), 'concentrazione ricalcolata diversa dal canale del toolbox'
display(T14.round(3))
cc = K['res'][..., IX['dominant_basin_concentration']]
fib = np.nan_to_num(TRU['npop']) > 0
fig_rampa(K, cc[MASK_B & fib], cc[MASK_B & ~fib], file=FIG / '10_rampa_modulazione.png'); plt.show()

## §17.2 — Il cono: dalla direzione del dizionario a quella vera

**Cosa guardare.** Attorno al nodo di partenza (stella), i candidati dei due livelli colorati per SSE (giallo =
errore più basso). Il cerchio rosso è il candidato scelto, la × la direzione vera. A destra l'errore angolare
prima e dopo. La scelta è controllata contro `refine_fiber_direction_cone` del toolbox.

In [ ]:
ij = voxel_di('WM sana', 5)
fig, INFO = fig_cono(K, ij, vera=dir_vere(ij)[0], file=FIG / '11_cono.png'); plt.show()
print({k: round(float(v), 3) for k, v in INFO.items()})

## §18 — Il crossing: la FF ri-stimata e i due tensori (MRDS)

**La FF del crossing.** In Stage A la penalità anisotropa schiaccia la FF dei crossing. Prima di stimare i
tensori, la FF viene ri-stimata con una NNLS sul solo "supporto" (i nodi vicini alle due direzioni rilevate, più
la griglia isotropa) con una ridge leggera κ(σ/S0)². A sinistra: la FF al variare di κ (κ = 0 tiene quella di
Stage A); la banda è su tutti i crossing del fantoccio, la linea nera su un voxel solo.

**I due tensori.** Con direzioni e frazioni fissate, si cercano (D∥, D⊥) delle due popolazioni: 3 iterazioni
alternate (una popolazione alla volta, per rompere la simmetria) e poi Levenberg–Marquardt. Al centro il costo
che scende e si ferma; a destra le quattro diffusività iterazione per iterazione (punteggiate quelle vere). La
traiettoria finisce esattamente sui tensori che il fit del toolbox ha scritto (controllato). Qui il fit ha il
cono dei crossing spento (come nella 1.8): dalla 1.9.0 dopo questo passo le direzioni lasciano la griglia (§18b).

In [ ]:
cand = [tuple(c) for c in np.argwhere((LAB == NOMI_SC.index('Crossing bil.')) & interno(LAB))]
err = [abs(K['res'][c][IX['fiber_fraction']] - TRU['FF'][c]) if K['res'][c][IX['n_fiber_populations']] == 2 else np.inf for c in cand]
ok = [c for c, e in zip(cand, err) if np.isfinite(e)]
ij = ok[int(np.argsort([e for e in err if np.isfinite(e)])[len(ok) // 2])]     # il voxel con l'errore mediano
fig, TRAI, INFO18 = fig_crossing(K, ij, verita=dict(FF=TRU['FF'][ij], AD=[1.6e-3, 1.6e-3], RD=[0.4e-3, 0.4e-3]),
                                  altri=ok, file=FIG / '12_crossing.png'); plt.show()
fin = TRAI.iloc[-1]
assert np.allclose([fin.AD1, fin.AD2, fin.RD1, fin.RD2], INFO18['AD_finale'] + INFO18['RD_finale'], rtol=1e-4), \
    'traiettoria MRDS diversa dai tensori del toolbox'
print({k: (np.round(v, 5) if not isinstance(v, list) else np.round(v, 5).tolist()) for k, v in INFO18.items()})

## §18b — Il cono per popolazione nei crossing (opzione `crossing_dir_refine`)

**Il problema.** Nei voxel a una fibra Stage C toglie la direzione dalla griglia del dizionario con il cono
(§17.2). Nei crossing no: le due direzioni restano sui nodi di Stage A, a qualche grado da quelle vere (la
spaziatura della griglia), e l'MRDS compensa l'errore di direzione con i tensori.

**L'opzione.** Dopo l'MRDS, una popolazione alla volta: candidati su due coni attorno alla direzione attuale
(gli stessi angoli del cono di Stage C); per ogni candidato si rifanno i due tensori con qualche passo di LM a
caldo, e si tiene il candidato che abbassa l'errore del modello a due popolazioni. Due giri, perché raffinare la
prima popolazione mentre la seconda è ancora sul suo nodo lascia una parte dell'errore. Un candidato che si
avvicina all'altra popolazione più della separazione minima è scartato.

**Cosa guardare.** Un crossing bilanciato (quello con l'errore mediano della griglia): a sinistra e al centro
le due popolazioni attorno alla direzione vera (×): stella = nodo di Stage A, cerchio = dopo il cono. A destra
l'errore mediano sui crossing della batteria e su 400 crossing orientati a caso. Le direzioni e i tensori ricalcolati sono **controllati** contro
quelli scritti dal fit con l'opzione accesa.

**Perché qui il cono sembra non servire.** Nella batteria le direzioni dei crossing cadono per costruzione
vicino ai nodi della griglia (errore della griglia ~4°, una popolazione dello sbilanciato a 1°): è già il
pavimento del rumore a questo SNR, e il cono non può scendere sotto. Dove la griglia è casualmente quasi
giusta e la popolazione è debole, il cono può anzi allontanarsi un po' (la stima segue il rumore). Il guadagno
vero si vede su orientamenti **casuali**, quando la direzione vera cade fra i nodi: la seconda tabella (crossing
bilanciati a 60–90° orientati a caso, stesso SNR) è il caso che conta per il tessuto.

Dalla 1.9.0 cono e test 2-contro-1 sono il default. Qui il fit di riferimento K (§14–18) li ha spenti, come nella
1.8: la §18 mostra l'MRDS sulle direzioni della griglia, la 18b accende il solo cono (K2), la 18c anche il test.

In [ ]:
HA_CROSSING = 'two_vs_one_threshold' in inspect.signature(DBSI_Adaptive).parameters
if not HA_CROSSING:
    print('toolbox senza le opzioni dei crossing (serve il branch crossing-cono-e-test21): 18b e 18c saltate')
else:
    K2 = intercetta_fit(P3, MB, MASK_B, np.full(MASK_B.shape, SIG_B), lam=(9.8, 0.015),
                        opzioni=dict(crossing_dir_refine=True, two_vs_one_threshold=None))   # stessi crossing di K
    CROSS = [n for n in NOMI_SC if n.startswith('Crossing')]
    E0 = errore_angolare(K['res'], TRU['dirs']); E1 = errore_angolare(K2['res'], TRU['dirs'])
    npk = lambda R, c: R[c][IX['n_fiber_populations']]
    cand = [tuple(c) for c in np.argwhere((LAB == NOMI_SC.index('Crossing bil.')) & interno(LAB))
            if npk(K['res'], tuple(c)) == 2 and npk(K2['res'], tuple(c)) == 2]
    ij = cand[int(np.argsort([np.nanmax(E0[c]) for c in cand])[len(cand) // 2])]
    C18 = cono_crossing_voxel(K, ij)
    fin = K2['res'][ij]
    d_tb = np.array([[fin[IX[f'dir{p}_{c}']] for c in 'xyz'] for p in (1, 2)])
    assert all(_ang(C18['dirs1'][k], d_tb[k]) < 0.05 for k in (0, 1)), 'cono ricalcolato diverso dal toolbox'
    assert np.allclose(C18['AD1'], [fin[IX['axial_diffusivity_pop1']], fin[IX['axial_diffusivity_pop2']]], rtol=1e-3), \
        'tensori dopo il cono diversi dal toolbox'
    righe = []
    for sc in CROSS:
        m_ = interno(LAB) & (LAB == NOMI_SC.index(sc))
        for br, E in (('griglia', E0), ('cono', E1)):
            for q in (1, 2):
                righe.append(dict(scenario=sc, popolazione=q, braccio=br, mediana=float(np.nanmedian(E[..., q - 1][m_]))))
    rc = []
    for br, opz in (('griglia', None), ('cono', dict(crossing_dir_refine=True))):
        e = crossing_casuali(P3, 400, SNR_FANTOCCIO, (9.8, 0.015), seed=7, opzioni=opz)
        for q in (0, 1):
            rc.append(dict(scenario='Casuali (bil.)', braccio=br, popolazione=q + 1, mediana=np.nanmedian(e[:, q]),
                           p90=np.nanpercentile(e[:, q], 90), visti_pct=100 * np.mean(np.isfinite(e[:, q]))))
    CAS18B = pd.DataFrame(rc)
    TAB18B = pd.concat([pd.DataFrame(righe), CAS18B[['scenario', 'popolazione', 'braccio', 'mediana']]], ignore_index=True)
    fig_cono_crossing(K, C18, dir_vere(ij), TAB18B, file=FIG / '12b_cono_crossing.png'); plt.show()
    display(TAB18B.pivot_table(index=['scenario', 'popolazione'], columns='braccio', values='mediana', sort=False).round(1))
    print(f'crossing bilanciati a orientamento casuale (400, SNR {SNR_FANTOCCIO}), errore angolare in gradi:')
    display(CAS18B.drop(columns='scenario').set_index(['popolazione', 'braccio']).round(1))

## §18c — Quante popolazioni: il test 2-contro-1 (opzione `two_vs_one_threshold`)

**Il problema.** Fino alla 1.8 il numero di popolazioni lo decideva Stage A: c'è un crossing se il secondo bacino
pesa almeno il 35% del primo (`min_peak_ratio`). È una soglia scelta a mano su distribuzioni che si
sovrappongono: un crossing sbilanciato vero (0.20 contro 0.45, rapporto 0.44) col rumore finisce sotto soglia
metà delle volte, e una fibra sola con un po' di rumore a volte la supera.

**Il test.** Ogni voxel con un secondo picco qualsiasi è un candidato: il fit stima **entrambi** i modelli, a una
e a due popolazioni (stesso Stage A), e li confronta sul modello finale di Stage D, prima della soglia di fibra:
$$\mathrm{stat21} = \frac{\mathrm{RSS}_{\text{1 pop}} - \mathrm{RSS}_{\text{2 pop}}}{\sigma_{\text{voxel}}^2}.$$
Se il voxel ha una fibra sola, il modello a due popolazioni spiega in più solo rumore: stat21 è piccola (circa
una χ² con pochi gradi di libertà, perché la seconda popolazione aggiunge una manciata di parametri). Con due
fibre vere la differenza cresce con l'SNR e con la seconda fibra. Si tiene il crossing se stat21 ≥ T (T = 12).
Il cono di 18b è acceso insieme: senza, il modello a due popolazioni parte svantaggiato (direzioni sulla
griglia) e il nullo diventa negativo.

**Cosa guardare.** A sinistra la statistica dei candidati con una e con due fibre vere. Al centro e a destra,
a tre SNR, la quota di voxel con due popolazioni stimate: deve **scendere** dove la fibra vera è una (falsi
crossing) e **salire** dove sono due (crossing trovati). La cella controlla che il toolbox abbia tenuto come
crossing esattamente i candidati con stat21 ≥ T.

In [ ]:
if HA_CROSSING:
    T21 = 12.0
    VERO = np.nan_to_num(TRU['npop']); INN_B = interno(LAB) & MASK_B
    rs, rn = [], []
    for snr in (15, 26, 40):
        if snr == SNR_FANTOCCIO:
            Mn, sg, Kd = MB, SIG_B, K
        else:
            Mn, sg = rumore_rician(A_b, snr, seed=1)
            Kd = intercetta_fit(P3, Mn, MASK_B, np.full(MASK_B.shape, sg), lam=(9.8, 0.015), opzioni=OPZ_18)
        Kt = intercetta_fit(P3, Mn, MASK_B, np.full(MASK_B.shape, sg), lam=(9.8, 0.015),
                            opzioni=dict(crossing_dir_refine=True, two_vs_one_threshold=T21))
        st = Kt['model'].two_vs_one_stat_
        npt = Kt['res'][..., IX['n_fiber_populations']]
        ok = np.isfinite(st) & np.isfinite(npt)
        assert np.array_equal(npt[ok] >= 2, st[ok] >= T21), 'decisione del toolbox diversa da stat21 >= T'
        for v in (1, 2):
            sel = INN_B & np.isfinite(st) & (VERO == v)
            rs += [dict(SNR=snr, vero=v, stat21=float(x)) for x in st[sel]]
            for nome, R in (('1.8', Kd['res']), ('test', Kt['res'])):
                n_ = np.nan_to_num(R[..., IX['n_fiber_populations']])[INN_B & (VERO == v)]
                rn.append(dict(SNR=snr, vero=v, fit=nome, due_pct=100 * float(np.mean(n_ >= 2))))
    STAT21, NPOP21 = pd.DataFrame(rs), pd.DataFrame(rn)
    fig_test21(STAT21, NPOP21, T21, SNR_FANTOCCIO, file=FIG / '12c_test_2contro1.png'); plt.show()
    display(NPOP21.pivot_table(index=['vero', 'fit'], columns='SNR', values='due_pct').round(1))
    display(STAT21.groupby(['SNR', 'vero']).stat21.describe(percentiles=[0.05, 0.5, 0.95]).round(1))

## §23 — Le mappe di incertezza, spiegate semplici

**Due parole: precisione e bias.** Immagina di misurare lo stesso voxel 300 volte, ognuna con un rumore
diverso. Le 300 stime formano una nuvola.
- **Quanto è larga la nuvola** è la *precisione*. L'errore standard (SE) che il toolbox scrive per ogni voxel
  stima proprio questa larghezza, **da una sola misura**: guarda quanto il segnale cambierebbe se il parametro
  cambiasse (se cambia molto, il parametro è ben determinato e la SE è piccola; se cambia poco, è mal
  determinato e la SE è grande) e la confronta con il rumore.
- **Dove sta il centro della nuvola rispetto alla verità** è il *bias*. La SE **non lo vede**: se lo stimatore
  sbaglia sempre nella stessa direzione, la nuvola è stretta ma spostata, e la mappa dice "preciso" anche
  quando è sistematicamente sbagliato.

**Cosa guardare.** Nella prima figura, la fibra sana: nuvola centrata sulla stella, ellisse (±1.96 SE) che la
contiene. Il crossing: la nuvola può essere spostata (freccia = bias). Nella seconda, la SE al variare di SNR e
protocollo: **la SE misura quello che il protocollo permette di sapere**. Per confrontare i protocolli dei 5P è
la parte che serve: a parità di tessuto, un protocollo con SE più grande ha mappe più rumorose.

In [ ]:
LAM = (R0['lambda_aniso'], R0['lambda_iso']) if FONTE.startswith('fantoccio') else (9.8, 0.015)
u = direzione(90, 20); v = direzione(90, 110)
I0 = dict(R=0.10, H=0.20, W=0.10)
casi = [('fibra sana', [(0.6, u, 1.7e-3, 0.4e-3)], 0.6, 0.4e-3),
        ('fibra demielinizzata', [(0.6, u, 1.5e-3, 0.8e-3)], 0.6, 0.8e-3),
        ('crossing 90°', [(0.3, u, 1.7e-3, 0.4e-3), (0.3, v, 1.7e-3, 0.4e-3)], 0.6, 0.4e-3)]
stime, ses, veri, tit = [], [], [], []
for nome, fibre, ff, rd in casi:
    res, se, fl, rr = fit_voxel_ripetuti(P3, fibre, I0, 300, SNR_FANTOCCIO, LAM, seed=11)
    stime.append(np.column_stack([res[:, IX['fiber_fraction']], res[:, IX['radial_diffusivity_weighted']] * 1e3]))
    ses.append((np.nanmedian(se[:, IX['fiber_fraction']]), np.nanmedian(se[:, IX['radial_diffusivity_weighted']]) * 1e3))
    veri.append((ff / (ff + 0.4), rd * 1e3)); tit.append(nome)
fig_bersaglio(stime, ses, veri, 'FF', 'RD pesata [e-3 mm²/s]', tit, file=FIG / '13_incertezza_bersaglio.png'); plt.show()

In [ ]:
righe = []
protos = {'P3': P3}
_root5p = Path(REALE['prep']).parents[3]          # .../DATA_5P/Processed
for P in (1, 2, 4, 5):
    base = _root5p / f'{P:06d}' / '20251210' / 'DWI_4s' / 'prep'
    fb, fv = base / f"{REALE['stem']}_corrected.bval", base / f"{REALE['stem']}_corrected.bvec"
    if fb.is_file() and fv.is_file():
        protos[f'P{P}'] = carica_protocollo(fb, fv)
print('protocolli:', list(protos))
for snr in (15, 26, 40, 60):
    for p, prot in protos.items():
        res, se, fl, rr = fit_voxel_ripetuti(prot, casi[0][1], I0, 100, snr, LAM, seed=3)
        for k, c, s in (('FF', 'fiber_fraction', 1), ('RD pesata [e-3]', 'radial_diffusivity_weighted', 1e3),
                        ('FA pesata', 'fiber_fa_weighted', 1)):
            righe.append(dict(protocollo=p, SNR=snr, metrica=k, se=float(np.nanmedian(se[:, IX[c]])) * s))
SEP = pd.DataFrame(righe)
fig_se_protocolli(SEP, file=FIG / '14_incertezza_snr.png'); plt.show()

**Il residuo e i flag.** Per ogni voxel il toolbox calcola anche `residual_over_sigma`: quanto il segnale
**non spiegato** dal modello è grande rispetto al rumore. Se modello e σ sono giusti vale ~1. Se è molto più
grande, il modello non descrive il voxel (o la mappa di rumore lo sottostima): il flag 16 si accende e la SE è
ottimista, perché presuppone un modello giusto. Qui: voxel che il modello descrive (diffusività isotrope uguali
ai centroidi) contro voxel con compartimenti che il modello non ha (diffusività isotrope lontane dai centroidi
e una terza fibra).

I flag in breve: **1** = la RD è sul limite inferiore (il dato non la determina, la SE descrive il modello senza
limite); **2** = la AD è sul limite; **4** = il modello è quasi non identificabile in quel voxel (la SE è poco
affidabile); **16** = il residuo è più grande del rumore (modello insufficiente o σ sottostimata).

In [ ]:
res_ok, se_ok, fl_ok, rr_ok = fit_voxel_ripetuti(P3, casi[0][1], I0, 300, SNR_FANTOCCIO, LAM, seed=21)
w = direzione(20, 60)
fibre_male = [(0.4, u, 1.7e-3, 0.4e-3), (0.2, v, 1.7e-3, 0.4e-3), (0.2, w, 1.7e-3, 0.4e-3)]
res_m, se_m, fl_m, rr_m = fit_voxel_ripetuti(P3, fibre_male, dict(R=0.05, H=0.10, W=0.05), 300, SNR_FANTOCCIO, LAM,
                                             seed=22, D=dict(R=0.05e-3, H=0.55e-3, W=2.2e-3))
dof = len(P3[0]) - 8
fig_residuo(rr_ok, rr_m, soglia_residuo(dof), file=FIG / '15_residuo_e_flag.png'); plt.show()
for nome, fl in (('modello giusto', fl_ok), ('modello insufficiente', fl_m)):
    print(f"{nome:22s} flag 1 {np.mean(fl & 1 > 0):.0%}  flag 2 {np.mean(fl & 2 > 0):.0%}  "
          f"flag 4 {np.mean(fl & 4 > 0):.0%}  flag 16 {np.mean(fl & 16 > 0):.0%}")

## Elenco delle figure

In [ ]:
for f in sorted(FIG.glob('*.png')):
    print(f.name)